Forecasting Algotrithm (Using XGBoost) For Hourly Gas Burns
Summer 2026
Clarissa J. Reynolds
Student Intern - Energy Trading


In [1]:
#Import necessary libraries
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

from datetime import timedelta
import os
from pathlib import Path
from sklearn.metrics import mean_squared_error
from mssql_python import connect
from nbdevAuto.functions import * 
import nbdevAuto.functions
import time 

In [2]:
# Database connection
SQL_CONNECTION_STRING = ("Server=HDQv1958;" "Database=allegro;" "Trusted_Connection=yes;" "Encrypt=yes;" "TrustServerCertificate=yes;")

conn = connect(SQL_CONNECTION_STRING)

In [3]:
#Getting site information and Daily gas burn 

def load_burn():

    query = """
SELECT t.trade, p.marketarea, q.begtime, q.energy, q.quantitystatus

FROM trade t, position p, ngquantity q

WHERE p.bepc_strategy = 'Burn' and t.trade = p.trade and t.tradestatus <> 'Void' and p.position = q.position and q.posstatus = 1
AND q.begtime >= DATEADD(year, -2, GETDATE()) AND q.begtime <= GETDATE()

ORDER BY q.begtime"""

    df = pd.read_sql(query, conn)
    #df["datetime"] = pd.to_datetime(df["datetime"])

    return df[["begtime", "marketarea", "energy"]]

gas_daily_df = load_burn()

#gas_daily_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 1.csv", index=False)

C:\Users\a102193\AppData\Local\Temp\ipykernel_37820\1453939605.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [2]:
#Caleb Data Load
gas_daily_df = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 1.csv")

In [3]:
#Adding the columns from the data that we need in the final dataset

#defining market areas as sites
gas_daily_df["marketarea"] = gas_daily_df["marketarea"].str.upper().str.strip()

gas_daily_df = gas_daily_df[
    ~gas_daily_df["marketarea"].isin(["BISON", "COTTAGE GROVE"])]

site_map = {"DEER CREEK": "DCS", "LANARK": "CGS", "LONSOME CREEK": "LCS",  "STATELINE": "PGS", "GROTON": "GGS", "CULBERTSON": "CGS"}

gas_daily_df["site"] = gas_daily_df["marketarea"].replace(site_map)



# create gas day

gas_daily_df["gas_day"] = pd.to_datetime(gas_daily_df["begtime"])


In [4]:

gas_daily_df["gas_day"] = pd.to_datetime(gas_daily_df["gas_day"]).dt.date


In [5]:
# aggregate on gas_day and renaming energy column to daily_gas_burn 
gas_daily_site = (gas_daily_df.groupby(["gas_day", "site"], as_index=False)["energy"].sum().rename(columns={"energy": "daily_gas_burn"}))


In [6]:
gas_daily_site.head()

,gas_day,site,daily_gas_burn
0,2024-07-23,CGS,13523.0
1,2024-07-23,DCS,46474.0
2,2024-07-23,GGS,11036.0
3,2024-07-23,LCS,42282.0
4,2024-07-23,PGS,33933.0


The next two queries are separate since PGS is aggregated by five minute intervals but the other generation sites are aggregated by hour

In [8]:
#Loading generation data for non-PGS sites

def load_generation_nonpgs():

    query = """
    SELECT begtime, loadshape, he1, he2, he3, he4, he5, he6, he7, he8, he9, he10, he11, he12, he13, he14, he15, he16, he17, he18, he19, he20, he21, he22, he23, he24
    FROM dbo.loadshapeprofile
    WHERE begtime >= DATEADD(year, -2, GETDATE()) AND begtime <= GETDATE()
      AND loadshape IN ('WAUE.BEPM.DCS1 - Net Generation',
        'WAUE.BEPM.LCS1 - Net Generation', 'WAUE.BEPM.LCS2 - Net Generation', 'WAUE.BEPM.LCS3 - Net Generation', 'WAUE.BEPM.LCS4 - Net Generation', 'WAUE.BEPM.LCS5 - Net Generation',
        'WAUE.BEPM.LCS6 - Net Generation', 'WAUE.BEPM.GGS1 - Net Generation', 'WAUE.BEPM.GGS2 - Net Generation', 'WAUE.BEPM.CULBERTSON1 - Net Generation')
    """

    return pd.read_sql(query, conn)

df_nonpgs = load_generation_nonpgs()

C:\Users\a102193\AppData\Local\Temp\ipykernel_37820\2149839047.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [7]:
df_nonpgs.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 2.csv", index=False)

NameError: name 'df_nonpgs' is not defined

In [8]:
#Caleb Data Load
df_nonpgs = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 2.csv")

In [9]:
#Loading generation data for PGS

def load_generation_pgs():

    query = """SELECT begtime, loadshape, SUM(he1)  AS he1, SUM(he2)  AS he2, SUM(he3)  AS he3, SUM(he4)  AS he4, SUM(he5)  AS he5, SUM(he6)  AS he6, SUM(he7)  AS he7,
    SUM(he8)  AS he8, SUM(he9)  AS he9, SUM(he10) AS he10, SUM(he11) AS he11, SUM(he12) AS he12, SUM(he13) AS he13, SUM(he14) AS he14, SUM(he15) AS he15, SUM(he16) AS he16,
    SUM(he17) AS he17, SUM(he18) AS he18, SUM(he19) AS he19, SUM(he20) AS he20, SUM(he21) AS he21, SUM(he22) AS he22, SUM(he23) AS he23, SUM(he24) AS he24

    FROM dbo.loadshapeprofile

    WHERE loadshape LIKE '%PGS%' AND loadshape LIKE '%- Net Generation-5m' AND begtime >= DATEADD(year, -2, GETDATE()) AND begtime <= GETDATE()

    GROUP BY begtime, loadshape

    ORDER BY begtime;"""

    return pd.read_sql(query, conn)

df_pgs = load_generation_pgs()

C:\Users\a102193\AppData\Local\Temp\ipykernel_37820\1754550502.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql(query, conn)


In [ ]:
#df_pgs.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 3.csv", index=False)

In [9]:
#Caleb Data Load
df_pgs = pd.read_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint sql 3.csv")

In [10]:
#Combining the data sources into one dataframe and mapping multiple generators to their corresponding sites 


df_load_generation = pd.concat([df_nonpgs, df_pgs], ignore_index=True)

# convert date using begtime
df_load_generation["begtime"] = pd.to_datetime(df_load_generation["begtime"])
df_load_generation["date"] = df_load_generation["begtime"].dt.date


# map site
df_load_generation["site"] = "N/A"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("DCS"), "site"] = "DCS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("LCS"), "site"] = "LCS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("PGS"), "site"] = "PGS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("GGS"), "site"] = "GGS"
df_load_generation.loc[df_load_generation["loadshape"].str.contains("CULBERTSON"), "site"] = "CGS"

In [11]:
df_load_generation.head()

,begtime,loadshape,he1,he2,he3,he4,he5,he6,he7,he8,...,he17,he18,he19,he20,he21,he22,he23,he24,date,site
0,2024-08-01,WAUE.BEPM.CULBERTSON1 - Net Generation,38.1,38.0,38.1,38.2,38.1,38.3,38.0,38.1,...,61.5,61.5,61.8,61.5,64.6,69.4,76.5,39.4,2024-08-01,CGS
1,2024-08-02,WAUE.BEPM.CULBERTSON1 - Net Generation,38.0,37.6,37.9,38.0,38.2,38.2,38.2,38.4,...,60.6,59.9,59.3,59.3,61.3,61.1,64.8,40.8,2024-08-02,CGS
2,2024-08-03,WAUE.BEPM.CULBERTSON1 - Net Generation,38.1,38.1,38.3,38.0,38.2,38.1,38.2,38.5,...,65.3,64.5,65.2,65.2,69.3,67.2,65.4,39.0,2024-08-03,CGS
3,2024-08-04,WAUE.BEPM.CULBERTSON1 - Net Generation,38.0,38.4,38.2,38.4,38.3,38.1,38.1,38.6,...,66.0,15.1,0.0,0.0,0.0,0.0,0.0,0.0,2024-08-04,CGS
4,2024-08-05,WAUE.BEPM.CULBERTSON1 - Net Generation,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2024-08-05,CGS


In [12]:
# melt function (python) instead of unpivot (SQL) which makes the data in long format instead of wide format

hour_cols = [column for column in df_load_generation.columns if column.startswith("he")]

hourly_df = df_load_generation.melt(id_vars=["begtime", "site", "loadshape"], value_vars=hour_cols, var_name="hour", value_name="hourly_mw")



In [22]:
hourly_df.head()
hourly_df[(hourly_df['site'] == 'DCS') ]

,begtime,site,loadshape,hour,hourly_mw
730,2024-08-01,DCS,WAUE.BEPM.DCS1 - Net Generation,he1,183.0
731,2024-08-02,DCS,WAUE.BEPM.DCS1 - Net Generation,he1,232.0
732,2024-08-03,DCS,WAUE.BEPM.DCS1 - Net Generation,he1,259.0
733,2024-08-04,DCS,WAUE.BEPM.DCS1 - Net Generation,he1,219.0
734,2024-08-05,DCS,WAUE.BEPM.DCS1 - Net Generation,he1,143.0
...,...,...,...,...,...
572269,2026-07-27,DCS,WAUE.BEPM.DCS1 - Net Generation,he24,236.0
572270,2026-07-28,DCS,WAUE.BEPM.DCS1 - Net Generation,he24,240.0
572271,2026-07-29,DCS,WAUE.BEPM.DCS1 - Net Generation,he24,151.0
572272,2026-07-30,DCS,WAUE.BEPM.DCS1 - Net Generation,he24,281.0


In [28]:
#standardize he to four characters (chronological)
hourly_df['hour'] =  np.where(hourly_df['hour'].str.len()==3, hourly_df['hour'].str[:2] + '0' + hourly_df['hour'].str[2:], hourly_df['hour'])

In [29]:
hourly_df.head()

,begtime,site,loadshape,hour,hourly_mw
12221,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0
235583,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he10,0.0
260401,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he11,0.0
285219,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he12,0.0
310037,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he13,0.0


In [30]:
#sort hourly_df
hourly_df =hourly_df.sort_values(by = ['site', 'loadshape', 'begtime', 'hour'], ascending=[False, False, True, True])
hourly_df[(hourly_df['site'] == 'DCS') ]

,begtime,site,loadshape,hour,hourly_mw
730,2024-08-01,DCS,WAUE.BEPM.DCS1 - Net Generation,he01,183.0
25548,2024-08-01,DCS,WAUE.BEPM.DCS1 - Net Generation,he02,207.0
50366,2024-08-01,DCS,WAUE.BEPM.DCS1 - Net Generation,he03,167.0
75184,2024-08-01,DCS,WAUE.BEPM.DCS1 - Net Generation,he04,223.0
100002,2024-08-01,DCS,WAUE.BEPM.DCS1 - Net Generation,he05,230.0
...,...,...,...,...,...
473001,2026-07-31,DCS,WAUE.BEPM.DCS1 - Net Generation,he20,278.0
497819,2026-07-31,DCS,WAUE.BEPM.DCS1 - Net Generation,he21,278.0
522637,2026-07-31,DCS,WAUE.BEPM.DCS1 - Net Generation,he22,278.0
547455,2026-07-31,DCS,WAUE.BEPM.DCS1 - Net Generation,he23,278.0


In [31]:
# Replace impossible generation spikes with previous hour's value
hourly_df.loc[hourly_df["hourly_mw"] > 400, "hourly_mw"] = np.nan

hourly_df['hourly_mw'] = hourly_df['hourly_mw'].ffill()


In [32]:
hourly_df.head(50000)

,begtime,site,loadshape,hour,hourly_mw
12221,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he01,0.0
37039,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he02,0.0
61857,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he03,0.0
86675,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he04,0.0
111493,2025-04-01,PGS,WAUE.BEPM.PGS5 - Net Generation-5m,he05,0.0
...,...,...,...,...,...
94908,2026-02-14,PGS,WAUE.BEPM.PGS35 - Net Generation-5m,he04,0.0
119726,2026-02-14,PGS,WAUE.BEPM.PGS35 - Net Generation-5m,he05,0.0
144544,2026-02-14,PGS,WAUE.BEPM.PGS35 - Net Generation-5m,he06,0.0
169362,2026-02-14,PGS,WAUE.BEPM.PGS35 - Net Generation-5m,he07,0.0


In [33]:
#9 records that are major outliers for a total of 10501976.59 MW and add back in 207.5 MW from forward fill and filled in daylight savings 


# extract hour number
hourly_df["hour_num"] = hourly_df["hour"].str[-2:].astype(int)

# build datetime
hourly_df["datetime"] = (pd.to_datetime(hourly_df["begtime"]) + pd.to_timedelta(hourly_df["hour_num"] - 0, unit="h"))

 
#hourly_df.head(48)
#hourly_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\Check 7-20 (1).csv", index=False)



In [34]:
# assign gas_day 
hourly_df["gas_day"] = (pd.to_datetime(hourly_df["datetime"]) - pd.Timedelta(hours=9)).dt.date

# extract hour #
hourly_df["hour"] = hourly_df["datetime"].dt.hour


In [35]:
#Reordering columns for visual 
hourly_df = hourly_df[["datetime","gas_day", "hour", "site", "loadshape", "hourly_mw"]]



In [36]:
#Hourly total by site
######################################################################################################################
hourly_site_gen_df = (hourly_df.groupby(["datetime", "gas_day", "hour", "site"], as_index=False)["hourly_mw"].sum().rename(columns={"hourly_mw": "hourly_site_gen_mw"}))

hourly_site_gen_df.head()

,datetime,gas_day,hour,site,hourly_site_gen_mw
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1
1,2024-08-01 01:00:00,2024-07-31,1,DCS,183.0
2,2024-08-01 01:00:00,2024-07-31,1,GGS,0.0
3,2024-08-01 01:00:00,2024-07-31,1,LCS,176.2
4,2024-08-01 01:00:00,2024-07-31,1,PGS,184.6


In [37]:
# Caleb Check
hourly_df[(hourly_df['site'] == 'DCS') & (hourly_df['datetime'] >= '2026-07-31')& (hourly_df['datetime'] <= '2026-08-01')]

,datetime,gas_day,hour,site,loadshape,hourly_mw
572272,2026-07-31 00:00:00,2026-07-30,0,DCS,WAUE.BEPM.DCS1 - Net Generation,281.0
1459,2026-07-31 01:00:00,2026-07-30,1,DCS,WAUE.BEPM.DCS1 - Net Generation,286.0
26277,2026-07-31 02:00:00,2026-07-30,2,DCS,WAUE.BEPM.DCS1 - Net Generation,278.0
51095,2026-07-31 03:00:00,2026-07-30,3,DCS,WAUE.BEPM.DCS1 - Net Generation,270.0
75913,2026-07-31 04:00:00,2026-07-30,4,DCS,WAUE.BEPM.DCS1 - Net Generation,226.0
100731,2026-07-31 05:00:00,2026-07-30,5,DCS,WAUE.BEPM.DCS1 - Net Generation,160.0
125549,2026-07-31 06:00:00,2026-07-30,6,DCS,WAUE.BEPM.DCS1 - Net Generation,156.0
150367,2026-07-31 07:00:00,2026-07-30,7,DCS,WAUE.BEPM.DCS1 - Net Generation,140.0
175185,2026-07-31 08:00:00,2026-07-30,8,DCS,WAUE.BEPM.DCS1 - Net Generation,141.0
200003,2026-07-31 09:00:00,2026-07-31,9,DCS,WAUE.BEPM.DCS1 - Net Generation,169.0


In [39]:
#Daily total by site


daily_site_gen_df = (hourly_df.groupby(["gas_day", "site"], as_index=False)["hourly_mw"].sum().rename(columns={"hourly_mw": "daily_site_gen_mw"}))
daily_site_gen_df.head()


,gas_day,site,daily_site_gen_mw
0,2024-07-31,CGS,304.9
1,2024-07-31,DCS,1611.0
2,2024-07-31,GGS,5.6
3,2024-07-31,LCS,1385.2
4,2024-07-31,PGS,1326.7


In [40]:
hourly_site_gen_df["gas_day"] = pd.to_datetime( hourly_site_gen_df["gas_day"])

gas_daily_site["gas_day"] = pd.to_datetime(gas_daily_site["gas_day"])

daily_site_gen_df["gas_day"] = pd.to_datetime(daily_site_gen_df["gas_day"])

In [41]:
# Add gas burn data
merged_df = hourly_site_gen_df.merge(gas_daily_site, on=["gas_day", "site"], how="left")


In [42]:
#caleb Check
merged_df[(merged_df['gas_day']=='2026-06-01') & (merged_df['site']=='PGS')]

,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn
80324,2026-06-01 09:00:00,2026-06-01,9,PGS,424.6,101141.0
80329,2026-06-01 10:00:00,2026-06-01,10,PGS,405.8,101141.0
80334,2026-06-01 11:00:00,2026-06-01,11,PGS,416.7,101141.0
80339,2026-06-01 12:00:00,2026-06-01,12,PGS,380.9,101141.0
80344,2026-06-01 13:00:00,2026-06-01,13,PGS,527.6,101141.0
80349,2026-06-01 14:00:00,2026-06-01,14,PGS,698.8,101141.0
80354,2026-06-01 15:00:00,2026-06-01,15,PGS,718.4,101141.0
80359,2026-06-01 16:00:00,2026-06-01,16,PGS,699.7,101141.0
80364,2026-06-01 17:00:00,2026-06-01,17,PGS,710.8,101141.0
80369,2026-06-01 18:00:00,2026-06-01,18,PGS,689.6,101141.0


In [43]:
print(hourly_site_gen_df["gas_day"].min())
print(hourly_site_gen_df["gas_day"].max())

print(gas_daily_site["gas_day"].min())
print(gas_daily_site["gas_day"].max())

print(daily_site_gen_df["gas_day"].min())
print(daily_site_gen_df["gas_day"].max())

2024-07-31 00:00:00
2026-07-31 00:00:00
2024-07-23 00:00:00
2026-07-22 00:00:00
2024-07-31 00:00:00
2026-07-31 00:00:00


In [44]:
test = hourly_site_gen_df.merge(
    gas_daily_site,
    on=["gas_day", "site"],
    how="left"
)

print(test["daily_gas_burn"].notna().sum())
print(len(test))

86560
87600


In [45]:
missing = (
    hourly_site_gen_df[["gas_day", "site"]]
    .drop_duplicates()
    .merge(
        gas_daily_site[["gas_day", "site"]],
        on=["gas_day", "site"],
        how="left",
        indicator=True
    )
)

print(
    missing[missing["_merge"] == "left_only"]
    .head(20)
)

        gas_day site     _merge
3610 2026-07-23  CGS  left_only
3611 2026-07-23  DCS  left_only
3612 2026-07-23  GGS  left_only
3613 2026-07-23  LCS  left_only
3614 2026-07-23  PGS  left_only
3615 2026-07-24  CGS  left_only
3616 2026-07-24  DCS  left_only
3617 2026-07-24  GGS  left_only
3618 2026-07-24  LCS  left_only
3619 2026-07-24  PGS  left_only
3620 2026-07-25  CGS  left_only
3621 2026-07-25  DCS  left_only
3622 2026-07-25  GGS  left_only
3623 2026-07-25  LCS  left_only
3624 2026-07-25  PGS  left_only
3625 2026-07-26  CGS  left_only
3626 2026-07-26  DCS  left_only
3627 2026-07-26  GGS  left_only
3628 2026-07-26  LCS  left_only
3629 2026-07-26  PGS  left_only


In [46]:
merged_df = merged_df.merge(daily_site_gen_df, on =["gas_day", "site"], how="left")
merged_df.head(10)


,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1,12340.0,304.9
1,2024-08-01 01:00:00,2024-07-31,1,DCS,183.0,43439.0,1611.0
2,2024-08-01 01:00:00,2024-07-31,1,GGS,0.0,9918.0,5.6
3,2024-08-01 01:00:00,2024-07-31,1,LCS,176.2,41713.0,1385.2
4,2024-08-01 01:00:00,2024-07-31,1,PGS,184.6,38470.0,1326.7
5,2024-08-01 02:00:00,2024-07-31,2,CGS,38.0,12340.0,304.9
6,2024-08-01 02:00:00,2024-07-31,2,DCS,207.0,43439.0,1611.0
7,2024-08-01 02:00:00,2024-07-31,2,GGS,0.0,9918.0,5.6
8,2024-08-01 02:00:00,2024-07-31,2,LCS,167.8,41713.0,1385.2
9,2024-08-01 02:00:00,2024-07-31,2,PGS,166.9,38470.0,1326.7


In [47]:
#sort merge_df chronological by site
merged_df = merged_df.sort_values(by=["site", "datetime"])

merged_df.head()

,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1,12340.0,304.9
5,2024-08-01 02:00:00,2024-07-31,2,CGS,38.0,12340.0,304.9
10,2024-08-01 03:00:00,2024-07-31,3,CGS,38.1,12340.0,304.9
15,2024-08-01 04:00:00,2024-07-31,4,CGS,38.2,12340.0,304.9
20,2024-08-01 05:00:00,2024-07-31,5,CGS,38.1,12340.0,304.9


In [48]:
merged_df[merged_df['daily_gas_burn'].isna()]

,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
86560,2026-07-23 09:00:00,2026-07-23,9,CGS,0.0,NaN,0.0
86565,2026-07-23 10:00:00,2026-07-23,10,CGS,0.0,NaN,0.0
86570,2026-07-23 11:00:00,2026-07-23,11,CGS,0.0,NaN,0.0
86575,2026-07-23 12:00:00,2026-07-23,12,CGS,0.0,NaN,0.0
86580,2026-07-23 13:00:00,2026-07-23,13,CGS,0.0,NaN,0.0
...,...,...,...,...,...,...,...
87579,2026-07-31 20:00:00,2026-07-31,20,PGS,0.0,NaN,0.0
87584,2026-07-31 21:00:00,2026-07-31,21,PGS,0.0,NaN,0.0
87589,2026-07-31 22:00:00,2026-07-31,22,PGS,0.0,NaN,0.0
87594,2026-07-31 23:00:00,2026-07-31,23,PGS,0.0,NaN,0.0


In [49]:
merged_df = merged_df[merged_df["daily_gas_burn"].notna()]

merged_df.head()

,datetime,gas_day,hour,site,hourly_site_gen_mw,daily_gas_burn,daily_site_gen_mw
0,2024-08-01 01:00:00,2024-07-31,1,CGS,38.1,12340.0,304.9
5,2024-08-01 02:00:00,2024-07-31,2,CGS,38.0,12340.0,304.9
10,2024-08-01 03:00:00,2024-07-31,3,CGS,38.1,12340.0,304.9
15,2024-08-01 04:00:00,2024-07-31,4,CGS,38.2,12340.0,304.9
20,2024-08-01 05:00:00,2024-07-31,5,CGS,38.1,12340.0,304.9


In [50]:
print(merged_df.columns.tolist())

['datetime', 'gas_day', 'hour', 'site', 'hourly_site_gen_mw', 'daily_gas_burn', 'daily_site_gen_mw']


In [51]:
#Caleb Check
print(merged_df.shape)

# 17424 rows ~20%. This is the only one that would cause a div by 0 error. Of these only 2208 are zero with a daily gas burn not 0.
# Is it possible to have daily burn but no measured site generation? Should we just divide by 24 and assume the burn occured evenly throughout the day?
print(merged_df[merged_df['daily_site_gen_mw'] == 0].shape) 

# I'm not sure this one should be replaced since it won't cause an error
print(merged_df[merged_df['hourly_site_gen_mw'] == 0].shape) #29896 rows ~34%

# Seems like a lot to replace
merged_df.to_csv('./output-data/merged_df check.csv', index=False)

(86560, 7)
(17400, 7)
(29771, 7)


In [ ]:
# not dividing by 0
# merged_df["hourly_site_gen_mw"] = (
merged_df["hourly_site_gen_mw"].replace(0, np.nan)
merged_df["daily_site_gen_mw"].replace(0, np.nan)

merged_df["hourly_gas_burn"] = ((merged_df["daily_gas_burn"] / merged_df["daily_site_gen_mw"]) * merged_df["hourly_site_gen_mw"]) 

merged_df.head()
#merged_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\Check 7-23 (1).csv", index=False)


In [ ]:
# final output of allegro data
result_df = merged_df[[ "datetime", "gas_day", "site", "hourly_site_gen_mw",  "daily_site_gen_mw", "daily_gas_burn", "hourly_gas_burn"]]


#math for daily MMBtu per MWh
result_df["gas_per_mw"] = (result_df["daily_gas_burn"] / result_df["daily_site_gen_mw"])

result_df.head(10)


,datetime,gas_day,site,hourly_site_gen_mw,daily_site_gen_mw,daily_gas_burn,hourly_gas_burn,gas_per_mw
0,2024-07-24 01:00:00,2024-07-23,CGS,38.0,305.0,13523.0,1684.832787,44.337705
5,2024-07-24 02:00:00,2024-07-23,CGS,38.1,305.0,13523.0,1689.266557,44.337705
10,2024-07-24 03:00:00,2024-07-23,CGS,37.9,305.0,13523.0,1680.399016,44.337705
15,2024-07-24 04:00:00,2024-07-23,CGS,38.1,305.0,13523.0,1689.266557,44.337705
20,2024-07-24 05:00:00,2024-07-23,CGS,38.0,305.0,13523.0,1684.832787,44.337705
25,2024-07-24 06:00:00,2024-07-23,CGS,38.3,305.0,13523.0,1698.134098,44.337705
30,2024-07-24 07:00:00,2024-07-23,CGS,38.3,305.0,13523.0,1698.134098,44.337705
35,2024-07-24 08:00:00,2024-07-23,CGS,38.3,305.0,13523.0,1698.134098,44.337705
40,2024-07-24 09:00:00,2024-07-24,CGS,38.4,1215.8,13343.0,421.427208,10.974667
45,2024-07-24 10:00:00,2024-07-24,CGS,38.0,1215.8,13343.0,417.037342,10.974667


In [45]:
#Caleb Check on null values
result_df.info()

<class 'pandas.DataFrame'>
Index: 87520 entries, 0 to 87519
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            87520 non-null  datetime64[us]
 1   gas_day             87520 non-null  datetime64[s] 
 2   site                87520 non-null  str           
 3   hourly_site_gen_mw  87520 non-null  float64       
 4   daily_site_gen_mw   87520 non-null  float64       
 5   daily_gas_burn      87520 non-null  float64       
 6   hourly_gas_burn     70096 non-null  float64       
 7   gas_per_mw          72304 non-null  float64       
dtypes: datetime64[s](1), datetime64[us](1), float64(5), str(1)
memory usage: 6.0 MB


In [46]:


export_df = result_df.copy()

#making a calendar date separate from gas day
export_df["date"] = export_df["datetime"].dt.date

export_df["gas_per_mw"] = (export_df["daily_gas_burn"] / export_df["daily_site_gen_mw"]).replace([np.inf, - np.inf], np.nan)

export_df = export_df[["datetime", "date", "gas_day", "site", "hourly_site_gen_mw", "daily_site_gen_mw", "daily_gas_burn", "hourly_gas_burn", "gas_per_mw"]]
export_df = export_df.sort_values(["site", "gas_day", "datetime"])


In [ ]:
#Caleb Record Count for the following code chunk
print(export_df.shape[0] - export_df[(export_df["daily_site_gen_mw"] > -1) & (export_df["gas_per_mw"] > 5) & (export_df["gas_per_mw"] < 20)].shape[0]) #23584 records removed

export_df.to_csv(r"./output-data/export", index=False)


23584

In [48]:
# I was trying to clean values but I am not sure what the numbers/parameter should be 
# Caleb Question: What is the purpose of filtering the DataFrame with these specific conditions? Are these thresholds based on domain knowledge or are they arbitrary?
# I'm always hesitant to remove data if it is correct, even if it is an outlier

clean_df = export_df[(export_df["daily_site_gen_mw"] > -1) & (export_df["gas_per_mw"] > 5) & (export_df["gas_per_mw"] < 20)]



export_df["gas_per_mw"] = pd.to_numeric(export_df["gas_per_mw"], errors="coerce")




In [ ]:
#Adding in an export to csv to output the data thus far

result_df = merged_df[["datetime", "gas_day", "site", "daily_site_gen_mw", "hourly_site_gen_mw", "daily_gas_burn", "hourly_gas_burn"]]

#commenting out to avoid overwriting
#result_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\allegro_data.csv", index=False)

In [52]:
# Caleb Export
result_df.to_csv(r"./output-data/result_df check.csv", index=False) 



In [ ]:
#Importing the Unit Availability data from Excel and CSV files from the G drive 
AVAILABILITY_FILES_XLSX = [
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 07.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 06.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 05.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 04.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 03.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 02.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2026\RT Unit availability 01.26.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 12.25.xlsx", #big issues with this file. I think the issue occured during Clarissa's manual process, so we can likely just fix without issues for others
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 11.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 10.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 09.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 08.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 07.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 06.25.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 05.25 - UPDATE.xlsx",
    r"G:\Trading\Market Operations\Unit availability\2025\RT Unit availability 04.25 - UPDATE.xlsx"]

AVAILABILITY_FILES_CSV = [
    r"G:\Trading\Market Operations\Unit availability\2025\dpm_BEPC_GROUPING_2025030100_2025033123 - March.csv",
    r"G:\Trading\Market Operations\Unit availability\2025\transposed_dpm_BEPC_GROUPING_2025020100_2025022823 feb.csv", #some smaller issues with this file as well. Looks like there are two 'extra' days of blank data. Likely copy and paste error of some type
    r"G:\Trading\Market Operations\Unit availability\2025\transposed_dpm_BEPC_GROUPING_2025010100_2025013123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024120100_2024123123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024110100_2024113023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024100100_2024103123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024090100_2024093023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024080100_2024083123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024070100_2024073123.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024060100_2024063023.csv",
    r"G:\Trading\Market Operations\Unit availability\2024\dpm_BEPC_GROUPING_2024050100_2024053123.csv"]


In [53]:
def pull_unit_availability(excel_files, csv_files):


    excel_dfs = []
    for file in excel_files:
        df = pd.read_excel(file, sheet_name="Gas HEL Transposed")
        df["source_file"] = file
        excel_dfs.append(df)

    excel_combined = pd.concat(excel_dfs, ignore_index=True)

    csv_dfs = []
    for file in csv_files:
        df = pd.read_csv(file)
        df["source_file"] = file
        csv_dfs.append(df)

    csv_combined = pd.concat(csv_dfs, ignore_index=True)

    #Concat the csvs and the excel files
    combined_df = pd.concat([excel_combined, csv_combined], ignore_index=True)

    #Making all column names lowercase and remove spaces
    combined_df.columns = (combined_df.columns.str.strip().str.lower())

    #Making datetime column in datetime format
    combined_df["datetime"] = pd.to_datetime( combined_df["datetime"], errors="coerce")

    #Sorting chronologically
    combined_df = (combined_df.sort_values("datetime").reset_index(drop=True))

    return combined_df




In [54]:
#Load data from the excel and csv files 
df = pull_unit_availability(excel_files, csv_files)



C:\Users\A105158\AppData\Local\Temp\ipykernel_25904\3039412008.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["source_file"] = file


In [57]:
# Caleb Check
df.to_csv('./output-data/unit availability check 1.csv', index = False)
df.shape #(35531, 801) This has way too many columns


(35531, 801)

In [55]:

#this is to defragment and get the performance warning to go away
df = df.copy()

# clean columns
df.columns = df.columns.astype(str) 
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)


In [ ]:
# Caleb Check
df.shape #(35531, 801) This has way too many columns from the issue with 12.25 file

(35531, 801)

In [57]:
# clean columns
df.columns = df.columns.astype(str) 
df.columns = df.columns.str.strip().str.replace(r"\s+", " ", regex=True)


In [58]:
# Caleb check
print(df.shape)
df.dtypes # already a datetime

(35531, 801)


datetime                       datetime64[us]
cgs1 - high effective limit             int64
dcs1 - high effective limit             int64
ggs1 - high effective limit             int64
ggs2 - high effective limit             int64
                                    ...      
0.761                                 float64
0.762                                 float64
0.763                                 float64
0.764                                 float64
0.765                                 float64
Length: 801, dtype: object

In [ ]:
# Caleb Checking na in datetime
print(df['datetime'].isna().sum()) #15689... why are they na? => Most are from the 12.25 file. 48 are from the feb 2025 file as well

# 
df[df['datetime'].isna() & df['cgs1 - high effective limit'] != 0] # two records without a datetime but with HEL values. I think these are related to daylight savings time changes

15689


,datetime,cgs1 - high effective limit,dcs1 - high effective limit,ggs1 - high effective limit,ggs2 - high effective limit,lcs1 - high effective limit,lcs2 - high effective limit,lcs3 - high effective limit,lcs4 - high effective limit,lcs5 - high effective limit,...,0.756,0.757,0.758,0.759,0.760,0.761,0.762,0.763,0.764,0.765
35481,NaT,95,297,0,0,0,0,0.0,41.0,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35530,NaT,95,297,95,0,35,38,0.0,33.0,35,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
# datetime fix
df = df.rename(columns={"DateTime": "datetime"})
df["datetime"] = pd.to_datetime(df["datetime"].astype(str), errors="coerce")
df = df.dropna(subset=["datetime"])


In [62]:
#Caleb Check
print(df.shape)
df.dtypes # I don't think anything actually changed

(19842, 801)


datetime                       datetime64[us]
cgs1 - high effective limit             int64
dcs1 - high effective limit             int64
ggs1 - high effective limit             int64
ggs2 - high effective limit             int64
                                    ...      
0.761                                 float64
0.762                                 float64
0.763                                 float64
0.764                                 float64
0.765                                 float64
Length: 801, dtype: object

In [63]:
## CALEB NOTE: I think fixing the source files would remove the need for this step


#Finding all High Effective Limit availability columns 
#value_cols = [col for col in df.columns if isinstance(col, str) and "High Effective Limit" in col]
value_cols = [col for col in df.columns if isinstance(col, str) and "high effective limit" in col]


In [64]:
value_cols

['cgs1 - high effective limit',
 'dcs1 - high effective limit',
 'ggs1 - high effective limit',
 'ggs2 - high effective limit',
 'lcs1 - high effective limit',
 'lcs2 - high effective limit',
 'lcs3 - high effective limit',
 'lcs4 - high effective limit',
 'lcs5 - high effective limit',
 'lcs6 - high effective limit',
 'pgs1 - high effective limit',
 'pgs2 - high effective limit',
 'pgs3 - high effective limit',
 'pgs11 - high effective limit',
 'pgs12 - high effective limit',
 'pgs13 - high effective limit',
 'pgs14 - high effective limit',
 'pgs15 - high effective limit',
 'pgs16 - high effective limit',
 'pgs17 - high effective limit',
 'pgs18 - high effective limit',
 'pgs19 - high effective limit',
 'pgs20 - high effective limit',
 'pgs21 - high effective limit',
 'pgs22 - high effective limit',
 'pgs31 - high effective limit',
 'pgs32 - high effective limit',
 'pgs33 - high effective limit',
 'pgs34 - high effective limit',
 'pgs35 - high effective limit',
 'pgs36 - high effectiv

In [65]:
# Caleb Check
df[value_cols].sum().sum()

np.float64(18991616.34)

In [66]:
#Caleb check
df.head()

,datetime,cgs1 - high effective limit,dcs1 - high effective limit,ggs1 - high effective limit,ggs2 - high effective limit,lcs1 - high effective limit,lcs2 - high effective limit,lcs3 - high effective limit,lcs4 - high effective limit,lcs5 - high effective limit,...,0.756,0.757,0.758,0.759,0.760,0.761,0.762,0.763,0.764,0.765
0,2024-05-01 00:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-05-01 01:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-05-01 02:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-05-01 03:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2024-05-01 04:00:00,45,297,0,0,32,32,39.0,0.0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [67]:

# melt (wide data to long data)
availability = df.melt(id_vars=["datetime"], value_vars=value_cols, var_name="unit", value_name="availability_mw")

availability.head()


,datetime,unit,availability_mw
0,2024-05-01 00:00:00,cgs1 - high effective limit,45.0
1,2024-05-01 01:00:00,cgs1 - high effective limit,45.0
2,2024-05-01 02:00:00,cgs1 - high effective limit,45.0
3,2024-05-01 03:00:00,cgs1 - high effective limit,45.0
4,2024-05-01 04:00:00,cgs1 - high effective limit,45.0


In [ ]:
# Caleb Check
print(availability['availability_mw'].sum()) # same as previous check
print(availability.info()) # It looks like there are 104 null values in availability_mw
availability.to_csv(r"./output-data/unit availability check 2.csv", index=False)

18991616.34
<class 'pandas.DataFrame'>
RangeIndex: 654786 entries, 0 to 654785
Data columns (total 3 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   datetime         654786 non-null  datetime64[us]
 1   unit             654786 non-null  str           
 2   availability_mw  654682 non-null  float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 15.0 MB
None


In [ ]:
#Checkpoint #11
#df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint11.csv")

In [70]:
availability.dtypes

datetime           datetime64[us]
unit                          str
availability_mw           float64
dtype: object

In [73]:

#Making sure the values are numeric so we can do the utilization calc
availability["availability_mw"] = pd.to_numeric(availability["availability_mw"], errors="coerce")


In [74]:
# Caleb Check
# I don't think the previous step did anything
print(availability['availability_mw'].sum())
print(availability.info()) # It looks like there are 104 null values in availability_mw
availability.to_csv(r"./output-data/unit availability check 3.csv", index=False)

18991616.34
<class 'pandas.DataFrame'>
RangeIndex: 654786 entries, 0 to 654785
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   datetime         654786 non-null  datetime64[us]
 1   unit             654786 non-null  str           
 2   availability_mw  654682 non-null  float64       
 3   site             654786 non-null  str           
dtypes: datetime64[us](1), float64(1), str(2)
memory usage: 20.0 MB
None


In [75]:
# extract site from unit name
#availability["site"] = availability["unit"].str.extract(r"^(CGS|DCS|GGS|LCS|PGS)")
availability["site"] = availability["unit"].str.extract(r"^(cgs|dcs|ggs|lcs|pgs)")
availability["site"] = availability["site"].str.strip().str.upper()


In [76]:
#Caleb Check
availability.to_csv(r"./output-data/unit availability check 4.csv", index=False)
print(availability['site'].value_counts())
print(availability['availability_mw'].sum())
availability.head()

site
PGS    456366
LCS    119052
GGS     39684
CGS     19842
DCS     19842
Name: count, dtype: int64
18991616.34


,datetime,unit,availability_mw,site
0,2024-05-01 00:00:00,cgs1 - high effective limit,45.0,CGS
1,2024-05-01 01:00:00,cgs1 - high effective limit,45.0,CGS
2,2024-05-01 02:00:00,cgs1 - high effective limit,45.0,CGS
3,2024-05-01 03:00:00,cgs1 - high effective limit,45.0,CGS
4,2024-05-01 04:00:00,cgs1 - high effective limit,45.0,CGS


In [77]:
# aggregate to site level 
site_availability = (availability.groupby(["datetime", "site"], as_index=False)["availability_mw"].sum())
site_availability.to_csv(r"./output-data/site availability check 5 point 1.csv", index=False)
site_availability.head(10)

,datetime,site,availability_mw
0,2024-05-01 00:00:00,CGS,45.0
1,2024-05-01 00:00:00,DCS,297.0
2,2024-05-01 00:00:00,GGS,0.0
3,2024-05-01 00:00:00,LCS,143.0
4,2024-05-01 00:00:00,PGS,131.8
5,2024-05-01 01:00:00,CGS,45.0
6,2024-05-01 01:00:00,DCS,297.0
7,2024-05-01 01:00:00,GGS,0.0
8,2024-05-01 01:00:00,LCS,143.0
9,2024-05-01 01:00:00,PGS,131.8


In [78]:
#caleb check
site_availability.sort_values(by = ['site', 'datetime']).head()

,datetime,site,availability_mw
0,2024-05-01 00:00:00,CGS,45.0
5,2024-05-01 01:00:00,CGS,45.0
10,2024-05-01 02:00:00,CGS,45.0
15,2024-05-01 03:00:00,CGS,45.0
20,2024-05-01 04:00:00,CGS,45.0


In [ ]:
site_availability.to_csv(r"./output-data/site availability check 5.csv", index=False)
print(site_availability['site'].value_counts())
print(site_availability['availability_mw'].sum()) # still matches
site_availability.head()

site
CGS    18188
DCS    18188
GGS    18188
LCS    18188
PGS    18188
Name: count, dtype: int64
18991616.34


,datetime,site,availability_mw
0,2024-05-01,CGS,45.0
1,2024-05-01,DCS,297.0
2,2024-05-01,GGS,0.0
3,2024-05-01,LCS,143.0
4,2024-05-01,PGS,131.8


In [ ]:
#Checkpoint #12
#site_availability.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint12.csv")

In [80]:
#Caleb Check
print(site_availability.shape)
site_availability.head()

(90940, 3)


,datetime,site,availability_mw
0,2024-05-01,CGS,45.0
1,2024-05-01,DCS,297.0
2,2024-05-01,GGS,0.0
3,2024-05-01,LCS,143.0
4,2024-05-01,PGS,131.8


In [81]:
#Caleb Check
print(result_df.shape)
result_df.head()

(87520, 7)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787
5,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557
10,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016
15,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557
20,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787


In [82]:

# merge to the rest of the df so far
master_df = result_df.copy()

master_df = master_df.merge(site_availability, on=["datetime", "site"], how="left")
master_df.head()


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0


In [83]:
# Caleb Check
master_df.to_csv(r"./output-data/master_df check 6.csv", index=False)
print(master_df.shape)
master_df.head()


(87520, 8)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0


In [ ]:
# Added by Caleb
# Below is code to find examples where the generation doesn't align with the submitted availability
# Example 1: Where generation is greater than availability
master_df[(master_df['site']=='DCS') & (master_df['hourly_site_gen_mw'] > master_df['availability_mw'])] #203 records
master_df[(master_df['site']=='PGS') & (master_df['hourly_site_gen_mw'] > master_df['availability_mw'])] #3475 records
master_df[(master_df['site']=='LCS') & (master_df['hourly_site_gen_mw'] > master_df['availability_mw'])] #1223 records
master_df[(master_df['site']=='GGS') & (master_df['hourly_site_gen_mw'] > master_df['availability_mw'])] #645 records
master_df[(master_df['site']=='CGS') & (master_df['hourly_site_gen_mw'] > master_df['availability_mw'])] #961 records

hourly_site_gen_exceeds_unit_availability = master_df[(master_df['hourly_site_gen_mw'] > master_df['availability_mw'])].to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\hourly_site_gen_exceeds_unit_availability.csv", index=False)


# based on spot checking, these appear to be situations where a unit could be ramping down or the Unit Availability excel doesn't align with what is in PCI. Screenshots in OneNote
print(master_df[(master_df['hourly_site_gen_mw'] > master_df['availability_mw'])].shape)


(6527, 8)


In [ ]:
# Added by Caleb
# Where generation is greater than site maxes
cgs_gen_more_than_max = master_df[(master_df['site']=='CGS') & (master_df['hourly_site_gen_mw'] > 95)] #4 records
pgs_gen_more_than_max = master_df[(master_df['site']=='PGS') & (master_df['hourly_site_gen_mw'] > 803.4 )] #0 records
lcs_gen_more_than_max = master_df[(master_df['site']=='LCS') & (master_df['hourly_site_gen_mw'] > 270 )] #2 records
ggs_gen_more_than_max = master_df[(master_df['site']=='GGS') & (master_df['hourly_site_gen_mw'] > 190 )] #0 records
cgs_gen_more_than_max = master_df[(master_df['site']=='CGS') & (master_df['hourly_site_gen_mw'] > 95 )] #4 records

hourly_site_gen_exceeds_site_max = pd.concat([cgs_gen_more_than_max, pgs_gen_more_than_max, lcs_gen_more_than_max, ggs_gen_more_than_max, cgs_gen_more_than_max])
hourly_site_gen_exceeds_site_max.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\hourly_site_gen_exceeds_site_max.csv", index=False)


# most are pretty small
hourly_site_gen_exceeds_site_max.head(20)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw
3155,2024-12-02 12:00:00,2024-12-02,CGS,770.0,95.3,7189.0,889.755455,95.0
3393,2024-12-12 10:00:00,2024-12-12,CGS,2106.7,95.3,19607.0,886.954526,95.0
3405,2024-12-12 22:00:00,2024-12-12,CGS,2106.7,95.3,19607.0,886.954526,95.0
3532,2024-12-18 05:00:00,2024-12-17,CGS,1913.8,95.5,17249.0,860.737538,95.0
53362,2024-08-28 11:00:00,2024-08-28,LCS,4465.5,480.0,41639.0,4475.807860,161.0
67497,2026-04-09 10:00:00,2026-04-09,LCS,3461.6,278.6,33001.0,2656.019933,235.0
3155,2024-12-02 12:00:00,2024-12-02,CGS,770.0,95.3,7189.0,889.755455,95.0
3393,2024-12-12 10:00:00,2024-12-12,CGS,2106.7,95.3,19607.0,886.954526,95.0
3405,2024-12-12 22:00:00,2024-12-12,CGS,2106.7,95.3,19607.0,886.954526,95.0
3532,2024-12-18 05:00:00,2024-12-17,CGS,1913.8,95.5,17249.0,860.737538,95.0


In [ ]:
#Checkpoint #13
#master_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint13.csv")

In [89]:
# CALEB SKIPPED - What is this function for? Is it to replace the previous chunks above


#not sure how much is repeated here but every time I remove a line it stops working
def load_unit_availability_by_site():
    
    df = pull_unit_availability(excel_files, csv_files)

    #Change column names to strings and strip whitespace
    df.columns = df.columns.map(lambda x: str(x).strip())

    df = df.rename(columns={"DateTime": "datetime"})

    #Standardize datetime 
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")

    # Only keep HEL columns
    hel_cols = [c for c in df.columns if "high effective limit" in c.lower()]

    df[hel_cols] = df[hel_cols].apply(pd.to_numeric, errors="coerce")

    df_long = df.melt(id_vars="datetime", value_vars=hel_cols, var_name="unit", value_name="availability_mw")

    df_long["site"] = df_long["unit"].str.extract(r"^(cgs|dcs|ggs|lcs|pgs)", expand=False).str.upper()

    availability_by_site = (df_long.groupby(["datetime", "site"], as_index=False).agg({"availability_mw": "sum"}))

    return availability_by_site


In [90]:
# CALEB SKIPPED
site_df = load_unit_availability_by_site()

C:\Users\A105158\AppData\Local\Temp\ipykernel_25904\3039412008.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["source_file"] = file


In [91]:
# CALEB SKIPPED
site_df.head()

,datetime,site,availability_mw
0,2024-05-01,CGS,45.0
1,2024-05-01,DCS,297.0
2,2024-05-01,GGS,0.0
3,2024-05-01,LCS,143.0
4,2024-05-01,PGS,131.8


In [92]:
site_df['availability_mw'].sum()

np.float64(18991616.34)

In [93]:
# CALEB SKIPPED
print(site_df["datetime"].min())
print(site_df["datetime"].max())

2024-05-01 00:00:00
2026-07-01 23:00:00


In [95]:
# Caleb check
master_df.sort_values(by=['site', 'datetime']).head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0


In [ ]:
# CALEB SKIPPED - is the _actual column any different?
#merge data to master df
master_df = master_df.merge(site_df, on=["datetime", "site"], how="left", suffixes=("", "_actual"))
master_df = master_df.drop_duplicates(["datetime", "site"])


In [ ]:
#Caleb Checking to see if the _actual column is any different than the non actual column
print(master_df[(master_df['availability_mw'] != master_df['availability_mw_actual']) & ~(master_df['availability_mw'].isna())].shape) # shape = (0, 9) - zero mismatches when only comparing rows with actual data
master_df.head()

(0, 9)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,availability_mw_actual
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,40.0
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,40.0
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0,40.0
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,40.0
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,40.0


In [102]:
# CALEB SKIPPED
print(master_df["datetime"].min())
print(master_df["datetime"].max())

2024-07-24 01:00:00
2026-07-23 08:00:00


In [103]:
master_df.head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,availability_mw_actual
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,40.0
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,40.0
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0,40.0
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,40.0
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,40.0


In [ ]:
#calculate utilization rate
master_df["utilization"] = (master_df["hourly_site_gen_mw"] / master_df["availability_mw"])


In [ ]:
# Caleb Check
master_df.to_csv(r"./output-data/utilization check.csv", index=False)


In [ ]:
# Caleb Check
master_df[np.isinf(master_df['utilization'])] 
# 945 records with inf. These have generation but no availability
# Is there a better way to handle these situations. Based on the data the sites are being utilized, so setting to nan seems like we're losing information

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,availability_mw_actual,utilization
280,2024-08-04 17:00:00,2024-08-04,CGS,476.6,66.0,4294.0,594.637012,0.0,0.0,inf
281,2024-08-04 18:00:00,2024-08-04,CGS,476.6,15.1,4294.0,136.045741,0.0,0.0,inf
321,2024-08-06 10:00:00,2024-08-06,CGS,2.3,2.3,0.0,0.000000,0.0,0.0,inf
350,2024-08-07 15:00:00,2024-08-07,CGS,722.0,11.1,9233.0,141.947784,0.0,0.0,inf
845,2024-08-28 06:00:00,2024-08-27,CGS,1420.6,43.5,13512.0,413.749120,0.0,0.0,inf
...,...,...,...,...,...,...,...,...,...,...
77498,2025-05-31 19:00:00,2025-05-31,PGS,3268.9,145.4,84581.0,3762.145492,0.0,0.0,inf
77499,2025-05-31 20:00:00,2025-05-31,PGS,3268.9,145.3,84581.0,3759.558047,0.0,0.0,inf
77500,2025-05-31 21:00:00,2025-05-31,PGS,3268.9,144.6,84581.0,3741.445930,0.0,0.0,inf
77501,2025-05-31 22:00:00,2025-05-31,PGS,3268.9,145.2,84581.0,3756.970602,0.0,0.0,inf


In [107]:

master_df["utilization"] = master_df["utilization"].replace([np.inf, -np.inf], np.nan)
master_df["utilization"].head()

0    0.9500
1    0.9525
2    0.9475
3    0.9525
4    0.9500
Name: utilization, dtype: float64

In [ ]:
#Checkpoint #14
#master_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint14.csv")

In [111]:
#to make sure the variables for the api are on the local os 
# Caleb modified slightly to work with his .env file 
YES_USER = os.getenv("YES_USERNAME")
YES_PASS = os.getenv("YES_PASSWORD")

print("YES_USER loaded:", YES_USER is not None)
print("YES_PASS loaded:", YES_PASS is not None)


YES_USER loaded: True
YES_PASS loaded: True


In [ ]:
def pull_yes_forecast_historical(user, password, start_date, end_date):

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    df_forecast = []

    while start <= end:

        month_start = start.replace(day=1)
        month_end = month_start + pd.offsets.MonthEnd(1)

        if month_end > end:
            month_end = end

        print(f"Pulling forecast: {month_start.date()} ---> {month_end.date()}")

        url = ( "https://services.yesenergy.com/PS/rest/timeseries/multiple.json?agglevel=hour&timezone=CPT"
            f"&startdate={month_start.date()}"
            f"&enddate={month_end.date()}"
            "&items="
            "LOAD_FORECAST:10017060648,"
            "NET_LOAD_FORECAST_CURRENT:10017060648,"
            "NG_CAPACITY_OFFLINE:10017060648,"
            "COAL_CAPACITY_OFFLINE:10017060648,"
            "WINDFCST_HOURLY:10004185377,"
            "WINDFCST_HOURLY:10004185378,"
            "WINDFCST_HOURLY:10004185379,"
            "WINDFCST_HOURLY:10004185380,"
            "WINDFCST_HOURLY:10004185381,"
            "WSI_FC15_FEEL:10000355230,"
            "WSI_FC15_FEEL:10000355704,"
            "WSI_FC15_FEEL:10000356081,"
            "WSI_FC15_WIND:10000355230,"
            "WSI_FC15_WIND:10000355704" )

        response = requests.get(url, auth=(user, password), verify=False, timeout=120)
        response.raise_for_status()
        
        # processing after successful data pull
        df_chunk = pd.DataFrame(response.json())
       
        df_chunk.columns = df_chunk.columns.map(lambda x: str(x).strip())

        df_forecast.append(df_chunk)

        time.sleep(6)  

        start = month_end + timedelta(days=1)

    return pd.concat(df_forecast, ignore_index=True)



In [114]:
#call the API
df_forecast = pull_yes_forecast_historical( YES_USER, YES_PASS, start_date="2023-06-01", end_date = pd.to_datetime(result_df["datetime"]).max().date())


Pulling forecast: 2023-06-01 ---> 2023-06-30
Pulling forecast: 2023-07-01 ---> 2023-07-31
Pulling forecast: 2023-08-01 ---> 2023-08-31
Pulling forecast: 2023-09-01 ---> 2023-09-30
Pulling forecast: 2023-10-01 ---> 2023-10-31
Pulling forecast: 2023-11-01 ---> 2023-11-30
Pulling forecast: 2023-12-01 ---> 2023-12-31
Pulling forecast: 2024-01-01 ---> 2024-01-31
Pulling forecast: 2024-02-01 ---> 2024-02-29
Pulling forecast: 2024-03-01 ---> 2024-03-31
Pulling forecast: 2024-04-01 ---> 2024-04-30
Pulling forecast: 2024-05-01 ---> 2024-05-31
Pulling forecast: 2024-06-01 ---> 2024-06-30
Pulling forecast: 2024-07-01 ---> 2024-07-31
Pulling forecast: 2024-08-01 ---> 2024-08-31
Pulling forecast: 2024-09-01 ---> 2024-09-30
Pulling forecast: 2024-10-01 ---> 2024-10-31
Pulling forecast: 2024-11-01 ---> 2024-11-30
Pulling forecast: 2024-12-01 ---> 2024-12-31
Pulling forecast: 2025-01-01 ---> 2025-01-31
Pulling forecast: 2025-02-01 ---> 2025-02-28
Pulling forecast: 2025-03-01 ---> 2025-03-31
Pulling fo

In [119]:
# Caleb Check
print(df_forecast.shape)
print(df_forecast.info())
df_forecast.head()

(27576, 7)
<class 'pandas.DataFrame'>
RangeIndex: 27576 entries, 0 to 27575
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   datetime       27576 non-null  datetime64[us]
 1   load           27576 non-null  int64         
 2   net_load       27576 non-null  float64       
 3   wind           27576 non-null  float64       
 4   temperature    27576 non-null  float64       
 5   wind_speed     27576 non-null  float64       
 6   total_outages  27570 non-null  float64       
dtypes: datetime64[us](1), float64(5), int64(1)
memory usage: 1.5 MB
None


,datetime,load,net_load,wind,temperature,wind_speed,total_outages
0,2023-06-01 01:00:00,28065,11285.63,16779.37,68.480,5.90,12204.1
1,2023-06-01 02:00:00,27109,11246.52,15862.48,67.045,6.20,12204.1
2,2023-06-01 03:00:00,26469,10391.44,16077.56,65.750,5.90,12204.1
3,2023-06-01 04:00:00,25911,9448.65,16462.35,64.765,5.30,12764.1
4,2023-06-01 05:00:00,25752,11442.85,14309.15,60.070,7.75,13040.3


In [ ]:
#Checkpoint #13
#df_forecast.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Data Checks (csv numbered)\checkpoint13.csv")

In [117]:

def clean_yes_forecast(df):

    df = df_forecast.copy()

    #find datetime column
    datetime_col = [c for c in df.columns if "DATETIME" in c.upper()][0]

    df["datetime"] = pd.to_datetime(df[datetime_col], format="%m/%d/%Y %H:%M:%S", errors="coerce")

    #load 
    df["load"] = pd.to_numeric( df["SPPISO-East (LOAD_FORECAST)"], errors="coerce")

    #net load 
    df["net_load"] = pd.to_numeric(df["SPPISO-East (NET_LOAD_FORECAST_CURRENT)"], errors="coerce")

    #wind 
    wind_cols = [c for c in df.columns if "WINDFCST_HOURLY" in c]
    df["wind"] = df[wind_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

    #outages 
    df["outage_ng"] = pd.to_numeric(df["SPPISO-East (NG_CAPACITY_OFFLINE)"], errors="coerce")

    df["outage_coal"] = pd.to_numeric(df["SPPISO-East (COAL_CAPACITY_OFFLINE)"], errors="coerce")

    #temperature avg of the zones (ask about this)
    temp_cols = [c for c in df.columns if "WSI_FC15_FEEL" in c]
    df["temperature"] = df[temp_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

    #wind speed avg of the reserve zones (also ask)
    wind_speed_cols = [c for c in df.columns if "WSI_FC15_WIND" in c]
    df["wind_speed"] = df[wind_speed_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)

    #outage
    df["total_outages"] = df["outage_ng"] + df["outage_coal"]

    out = df[["datetime", "load", "net_load", "wind", "temperature", "wind_speed", "total_outages"]].dropna(subset=["datetime"])

    return out.sort_values("datetime").reset_index(drop=True)


In [118]:
#save the changes and get rid of forecast after for formatting
df_forecast = clean_yes_forecast(df_forecast)


forecast_columns = [columns for columns in master_df.columns if columns.endswith("_forecast")]

master_df = master_df.drop(columns=forecast_columns, errors="ignore")

#merge data to master df
master_df = master_df.merge(df_forecast, on="datetime", how="left", suffixes=("", "_forecast"))


In [120]:
# Caleb Check
print(df_forecast.shape)
print(df_forecast.info())
df_forecast.head()

(27576, 7)
<class 'pandas.DataFrame'>
RangeIndex: 27576 entries, 0 to 27575
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   datetime       27576 non-null  datetime64[us]
 1   load           27576 non-null  int64         
 2   net_load       27576 non-null  float64       
 3   wind           27576 non-null  float64       
 4   temperature    27576 non-null  float64       
 5   wind_speed     27576 non-null  float64       
 6   total_outages  27570 non-null  float64       
dtypes: datetime64[us](1), float64(5), int64(1)
memory usage: 1.5 MB
None


,datetime,load,net_load,wind,temperature,wind_speed,total_outages
0,2023-06-01 01:00:00,28065,11285.63,16779.37,68.480,5.90,12204.1
1,2023-06-01 02:00:00,27109,11246.52,15862.48,67.045,6.20,12204.1
2,2023-06-01 03:00:00,26469,10391.44,16077.56,65.750,5.90,12204.1
3,2023-06-01 04:00:00,25911,9448.65,16462.35,64.765,5.30,12764.1
4,2023-06-01 05:00:00,25752,11442.85,14309.15,60.070,7.75,13040.3


In [221]:
df_forecast['datetime'].max()

Timestamp('2026-07-24 00:00:00')

In [121]:

def pull_yes_actual_historical(user, password, start_date, end_date):

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    df_actual = []

    while start <= end:

        month_start = start.replace(day=1)
        month_end = month_start + pd.offsets.MonthEnd(1)

        if month_end > end:
            month_end = end

        print(f"Pulling actuals: {month_start.date()} ---> {month_end.date()}")

        url = ("https://services.yesenergy.com/PS/rest/timeseries/multiple.json?agglevel=hour&timezone=CPT"
            f"&startdate={month_start.date()}"
            f"&enddate={month_end.date()}"
            "&items="
            #day ahead close so just in actual
            "BIDCLOSE_LOAD_FORECAST:10017060648,"
            "NET_LOAD_FORECAST_BID_CLOSE:10017060648,"
            "NG_CAPACITY_OFFLINE:10017060648,"
            "COAL_CAPACITY_OFFLINE:10017060648,"
            "WINDGEN_HOURLY:10004185377,"
            "WINDGEN_HOURLY:10004185378,"
            "WINDGEN_HOURLY:10004185379,"
            "WINDGEN_HOURLY:10004185380,"
            "WINDGEN_HOURLY:10004185381,"
            "WSI_TRADER_FEELS_TEMP:10000355230,"
            "WSI_TRADER_FEELS_TEMP:10000355704,"
            "WSI_TRADER_FEELS_TEMP:10000356081,"
            "WSI_TRADER_WIND:10000355230,"
            "WSI_TRADER_WIND:10000355704")


        response = requests.get(url, auth=(user, password), verify=False, timeout=120)
        #response.raise_for_status()


        df_chunk = pd.DataFrame(response.json())
        df_chunk.columns = df_chunk.columns.map(lambda x: str(x).strip())

        df_actual.append(df_chunk)
        
        time.sleep(6)  

        start = month_end + timedelta(days=1)

    return pd.concat(df_actual, ignore_index=True)



In [122]:


def clean_yes_actual(df):
  
    df = df_actual.copy()

    datetime_col = [c for c in df.columns if "DATETIME" in c.upper()][0]

    df["datetime"] = pd.to_datetime( df[datetime_col], errors="coerce")


    # load actual
    df["load_actual"] = pd.to_numeric(df["SPPISO-East (BIDCLOSE_LOAD_FORECAST)"], errors="coerce")

    # net load actual
    df["net_load_actual"] = pd.to_numeric( df["SPPISO-East (NET_LOAD_FORECAST_BID_CLOSE)"], errors="coerce")

    # wind actual
    wind_cols = [c for c in df.columns if "WINDGEN_HOURLY" in c]
    wind_numeric = (df[wind_cols].apply(pd.to_numeric, errors="coerce"))

    wind_sum = wind_numeric.sum(axis=1, min_count=1)

    all_zero = (wind_numeric.fillna(0).sum(axis=1).eq(0))

    df["wind_actual"] = wind_sum.mask(all_zero)

    # outage actual
    df["outage_ng"] = pd.to_numeric(df["SPPISO-East (NG_CAPACITY_OFFLINE)"], errors="coerce")
    df["outage_coal"] = pd.to_numeric( df["SPPISO-East (COAL_CAPACITY_OFFLINE)"], errors="coerce")

    # temperature actual
    temp_cols = [c for c in df.columns if "WSI_TRADER_FEELS_TEMP" in c]
    temp_numeric = (df[temp_cols].apply(pd.to_numeric, errors="coerce"))

    temp_avg = temp_numeric.mean(axis=1)

    all_zero_temp = (temp_numeric.fillna(0).sum(axis=1).eq(0))

    df["temperature_actual"] = temp_avg.mask(all_zero_temp)

    # windspeed actual
    wind_speed_cols = [c for c in df.columns if "WSI_TRADER_WIND" in c]
    df["wind_speed_actual"] = df[wind_speed_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
    
    
    # outage actual
    df["total_outages"] = df["outage_ng"] + df["outage_coal"]

    out = df[["datetime", "load_actual", "net_load_actual", "wind_actual", "temperature_actual", "wind_speed_actual", "total_outages"]].dropna(subset=["datetime"])

    return out.sort_values("datetime").reset_index(drop=True)


In [123]:
#call the API
df_actual = pull_yes_actual_historical( YES_USER, YES_PASS, start_date="2023-06-01", end_date = pd.to_datetime(result_df["datetime"]).max().date())


Pulling actuals: 2023-06-01 ---> 2023-06-30
Pulling actuals: 2023-07-01 ---> 2023-07-31
Pulling actuals: 2023-08-01 ---> 2023-08-31
Pulling actuals: 2023-09-01 ---> 2023-09-30
Pulling actuals: 2023-10-01 ---> 2023-10-31
Pulling actuals: 2023-11-01 ---> 2023-11-30
Pulling actuals: 2023-12-01 ---> 2023-12-31
Pulling actuals: 2024-01-01 ---> 2024-01-31
Pulling actuals: 2024-02-01 ---> 2024-02-29
Pulling actuals: 2024-03-01 ---> 2024-03-31
Pulling actuals: 2024-04-01 ---> 2024-04-30
Pulling actuals: 2024-05-01 ---> 2024-05-31
Pulling actuals: 2024-06-01 ---> 2024-06-30
Pulling actuals: 2024-07-01 ---> 2024-07-31
Pulling actuals: 2024-08-01 ---> 2024-08-31
Pulling actuals: 2024-09-01 ---> 2024-09-30
Pulling actuals: 2024-10-01 ---> 2024-10-31
Pulling actuals: 2024-11-01 ---> 2024-11-30
Pulling actuals: 2024-12-01 ---> 2024-12-31
Pulling actuals: 2025-01-01 ---> 2025-01-31
Pulling actuals: 2025-02-01 ---> 2025-02-28
Pulling actuals: 2025-03-01 ---> 2025-03-31
Pulling actuals: 2025-04-01 --->

In [128]:
df_actual.info()
df_actual.head()

<class 'pandas.DataFrame'>
RangeIndex: 27576 entries, 0 to 27575
Data columns (total 20 columns):
 #   Column                                           Non-Null Count  Dtype 
---  ------                                           --------------  ----- 
 0   DATETIME                                         27576 non-null  str   
 1   SPPISO-East (BIDCLOSE_LOAD_FORECAST)             27576 non-null  str   
 2   SPPISO-East (NET_LOAD_FORECAST_BID_CLOSE)        27576 non-null  str   
 3   SPPISO-East (NG_CAPACITY_OFFLINE)                27570 non-null  str   
 4   SPPISO-East (COAL_CAPACITY_OFFLINE)              27570 non-null  str   
 5   RESERVE ZONE 1 (WINDGEN_HOURLY)                  27561 non-null  str   
 6   RESERVE ZONE 2 (WINDGEN_HOURLY)                  27561 non-null  str   
 7   RESERVE ZONE 3 (WINDGEN_HOURLY)                  27561 non-null  str   
 8   RESERVE ZONE 4 (WINDGEN_HOURLY)                  27561 non-null  str   
 9   RESERVE ZONE 5 (WINDGEN_HOURLY)                  2

,DATETIME,SPPISO-East (BIDCLOSE_LOAD_FORECAST),SPPISO-East (NET_LOAD_FORECAST_BID_CLOSE),SPPISO-East (NG_CAPACITY_OFFLINE),SPPISO-East (COAL_CAPACITY_OFFLINE),RESERVE ZONE 1 (WINDGEN_HOURLY),RESERVE ZONE 2 (WINDGEN_HOURLY),RESERVE ZONE 3 (WINDGEN_HOURLY),RESERVE ZONE 4 (WINDGEN_HOURLY),RESERVE ZONE 5 (WINDGEN_HOURLY),ND - Bismarck/Municipal (WSI_TRADER_FEELS_TEMP),ND - Fargo/Hector Field (WSI_TRADER_FEELS_TEMP),ND - Williston/Sloulin (WSI_TRADER_FEELS_TEMP),ND - Bismarck/Municipal (WSI_TRADER_WIND),ND - Fargo/Hector Field (WSI_TRADER_WIND),HOURENDING,MARKETDAY,PEAKTYPE,MONTH,YEAR
0,06/01/2023 01:00:00,28092,11697.25,7119.1,5085,1650.35,3191.708,1627.487,8045.607,1287.408,58,69,None,0,5.7,1,06/01/2023,None,JUNE,2023
1,06/01/2023 02:00:00,26869,10770.89,7119.1,5085,1508.173,3108.084,1456.827,8155.067,1472.342,58,70,None,0,9.2,2,06/01/2023,None,JUNE,2023
2,06/01/2023 03:00:00,26031,11484.51,7119.1,5085,1453.025,3175.017,1467.9,7478.217,1741.225,55,69,None,4.6,12.7,3,06/01/2023,None,JUNE,2023
3,06/01/2023 04:00:00,25548,12909.48,7679.1,5085,1232.967,3091.492,1355.074,7259.4,1689.458,56,66,None,3.4,8.1,4,06/01/2023,None,JUNE,2023
4,06/01/2023 05:00:00,25496,14232.5,7277.1,5763.2,1040.083,2984.922,1279.567,7355.633,1530.012,58,66,None,4.6,9.2,5,06/01/2023,None,JUNE,2023


In [124]:
#now cleaned
df_actual_clean = clean_yes_actual(df_actual)



#making sure if YES changes how they format datetime the code still runs
datetime_column = [columns for columns in df_actual.columns
    if "DATETIME" in columns.upper()][0]

columns_to_drop = [c for c in master_df.columns if c.endswith("_actual")]

master_df = master_df.drop(columns=columns_to_drop, errors="ignore")


In [126]:
# Caleb Check
df_actual.head()

,DATETIME,SPPISO-East (BIDCLOSE_LOAD_FORECAST),SPPISO-East (NET_LOAD_FORECAST_BID_CLOSE),SPPISO-East (NG_CAPACITY_OFFLINE),SPPISO-East (COAL_CAPACITY_OFFLINE),RESERVE ZONE 1 (WINDGEN_HOURLY),RESERVE ZONE 2 (WINDGEN_HOURLY),RESERVE ZONE 3 (WINDGEN_HOURLY),RESERVE ZONE 4 (WINDGEN_HOURLY),RESERVE ZONE 5 (WINDGEN_HOURLY),ND - Bismarck/Municipal (WSI_TRADER_FEELS_TEMP),ND - Fargo/Hector Field (WSI_TRADER_FEELS_TEMP),ND - Williston/Sloulin (WSI_TRADER_FEELS_TEMP),ND - Bismarck/Municipal (WSI_TRADER_WIND),ND - Fargo/Hector Field (WSI_TRADER_WIND),HOURENDING,MARKETDAY,PEAKTYPE,MONTH,YEAR
0,06/01/2023 01:00:00,28092,11697.25,7119.1,5085,1650.35,3191.708,1627.487,8045.607,1287.408,58,69,None,0,5.7,1,06/01/2023,None,JUNE,2023
1,06/01/2023 02:00:00,26869,10770.89,7119.1,5085,1508.173,3108.084,1456.827,8155.067,1472.342,58,70,None,0,9.2,2,06/01/2023,None,JUNE,2023
2,06/01/2023 03:00:00,26031,11484.51,7119.1,5085,1453.025,3175.017,1467.9,7478.217,1741.225,55,69,None,4.6,12.7,3,06/01/2023,None,JUNE,2023
3,06/01/2023 04:00:00,25548,12909.48,7679.1,5085,1232.967,3091.492,1355.074,7259.4,1689.458,56,66,None,3.4,8.1,4,06/01/2023,None,JUNE,2023
4,06/01/2023 05:00:00,25496,14232.5,7277.1,5763.2,1040.083,2984.922,1279.567,7355.633,1530.012,58,66,None,4.6,9.2,5,06/01/2023,None,JUNE,2023


In [127]:
# Caleb Check
df_actual_clean.info()
df_actual_clean.head()

<class 'pandas.DataFrame'>
RangeIndex: 27576 entries, 0 to 27575
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   datetime            27576 non-null  datetime64[us]
 1   load_actual         27576 non-null  int64         
 2   net_load_actual     27576 non-null  float64       
 3   wind_actual         27561 non-null  float64       
 4   temperature_actual  27539 non-null  float64       
 5   wind_speed_actual   27570 non-null  float64       
 6   total_outages       27570 non-null  float64       
dtypes: datetime64[us](1), float64(5), int64(1)
memory usage: 1.5 MB


,datetime,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages
0,2023-06-01 01:00:00,28092,11697.25,15802.560,63.5,2.85,12204.1
1,2023-06-01 02:00:00,26869,10770.89,15700.493,64.0,4.60,12204.1
2,2023-06-01 03:00:00,26031,11484.51,15315.384,62.0,8.65,12204.1
3,2023-06-01 04:00:00,25548,12909.48,14628.391,61.0,5.75,12764.1
4,2023-06-01 05:00:00,25496,14232.50,14190.217,62.0,6.90,13040.3


In [ ]:
# Caleb Null Check
# flipping through the different columns it seems to just be noise. Not seeing anything systematic/programmatic about the NANs
df_actual_clean[df_actual_clean['wind_speed_actual'].isnull()]

,datetime,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages
705,2023-06-30 10:00:00,38842,29382.194167,8328.838,NaN,NaN,7622.0
3769,2023-11-05 02:00:00,25530,16836.190000,9339.426,NaN,NaN,NaN
8065,2024-05-02 02:00:00,26598,3270.140000,18621.047,NaN,NaN,22919.4
8066,2024-05-02 03:00:00,26079,3223.480000,18059.577,NaN,NaN,22919.4
12505,2024-11-03 02:00:00,25840,3621.550000,13094.613,NaN,NaN,NaN
21241,2025-11-02 02:00:00,28202,14597.960000,15654.493,NaN,NaN,NaN


In [136]:
master_df.head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,net_load,wind,temperature,wind_speed,total_outages
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,33629.0,24945.22,8683.78,71.900000,6.5,7362.56
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,32426.0,23252.87,9173.13,70.640000,6.2,7362.56
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0,0.9475,31385.0,22875.04,8509.96,69.740000,6.2,7362.56
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,30679.0,21948.26,8730.74,68.840000,5.9,7362.56
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,30485.0,22043.42,8441.58,66.923333,6.5,7784.06


In [139]:
# Caleb Check
print(master_df.shape)
master_df.head()

(87530, 15)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,net_load,wind,temperature,wind_speed,total_outages
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,33629.0,24945.22,8683.78,71.900000,6.5,7362.56
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,32426.0,23252.87,9173.13,70.640000,6.2,7362.56
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0,0.9475,31385.0,22875.04,8509.96,69.740000,6.2,7362.56
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,30679.0,21948.26,8730.74,68.840000,5.9,7362.56
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,30485.0,22043.42,8441.58,66.923333,6.5,7784.06


In [140]:
#merge data to master df
master_df = master_df.merge(df_actual_clean, on="datetime", how="left", suffixes=("", "_actual"))

In [141]:
# Caleb Check
print(master_df.shape)
master_df.head()

(87550, 21)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,wind,temperature,wind_speed,total_outages,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages_actual
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,33629.0,...,8683.78,71.900000,6.5,7362.56,33707.0,24844.94,8511.573,69.666667,5.15,7362.56
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,32426.0,...,9173.13,70.640000,6.2,7362.56,32270.0,22600.18,8509.053,68.000000,5.70,7362.56
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0,0.9475,31385.0,...,8509.96,69.740000,6.2,7362.56,31254.0,21481.27,8834.246,67.666667,6.30,7362.56
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,30679.0,...,8730.74,68.840000,5.9,7362.56,30549.0,21074.88,8839.246,67.333333,6.30,7362.56
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,30485.0,...,8441.58,66.923333,6.5,7784.06,30424.0,21411.54,8194.537,66.000000,6.30,7784.06


In [149]:
# Caleb Check on duplicates that would be getting dropped
# these are all in the first week of November and they appear to be from daylight savings and shifting clocks back one hour each year
master_df[master_df.duplicated(['datetime', 'site'])].sort_values(by = ['site', 'datetime']).shape
master_df[master_df.duplicated(['datetime', 'site'])].sort_values(by = ['site', 'datetime']).head()

,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,wind,temperature,wind_speed,total_outages,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages_actual
2450,2024-11-03 02:00:00,2024-11-02,CGS,0.0,0.0,0.0,NaN,95.0,0.0,25795.0,...,13163.06,40.270000,12.600,NaN,25738.0,4262.35,13132.619,44.0,15.55,22870.94
2451,2024-11-03 02:00:00,2024-11-02,CGS,0.0,0.0,0.0,NaN,95.0,0.0,25385.0,...,12735.96,40.270000,12.600,22870.94,25840.0,3621.55,13094.613,NaN,NaN,NaN
2452,2024-11-03 02:00:00,2024-11-02,CGS,0.0,0.0,0.0,NaN,95.0,0.0,25385.0,...,12735.96,40.270000,12.600,22870.94,25738.0,4262.35,13132.619,44.0,15.55,22870.94
11189,2025-11-02 02:00:00,2025-11-01,CGS,0.0,0.0,0.0,NaN,190.0,0.0,28199.0,...,15768.81,35.893333,13.675,NaN,27963.0,13422.84,16581.340,34.0,10.90,18814.90
11190,2025-11-02 02:00:00,2025-11-01,CGS,0.0,0.0,0.0,NaN,190.0,0.0,28190.0,...,17219.43,35.893333,13.675,18814.90,28202.0,14597.96,15654.493,NaN,NaN,NaN


In [ ]:
print(f'dimensions before dropping duplicates: {master_df.shape}')
master_df = master_df.drop_duplicates(["datetime", "site"]) #drops 30 records.
print(f'dimensions after dropping duplicates: {master_df.shape}')

dimensions before dropping duplicates: (87550, 21)
dimensions after dropping duplicates: (87520, 21)


In [156]:
print(master_df.columns)
master_df.head()

Index(['datetime', 'gas_day', 'site', 'daily_site_gen_mw',
       'hourly_site_gen_mw', 'daily_gas_burn', 'hourly_gas_burn',
       'availability_mw', 'utilization', 'load', 'net_load', 'wind',
       'temperature', 'wind_speed', 'total_outages', 'load_actual',
       'net_load_actual', 'wind_actual', 'temperature_actual',
       'wind_speed_actual', 'total_outages_actual'],
      dtype='str')


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,wind,temperature,wind_speed,total_outages,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages_actual
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,33629.0,...,8683.78,71.900000,6.5,7362.56,33707.0,24844.94,8511.573,69.666667,5.15,7362.56
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,32426.0,...,9173.13,70.640000,6.2,7362.56,32270.0,22600.18,8509.053,68.000000,5.70,7362.56
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0,0.9475,31385.0,...,8509.96,69.740000,6.2,7362.56,31254.0,21481.27,8834.246,67.666667,6.30,7362.56
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,30679.0,...,8730.74,68.840000,5.9,7362.56,30549.0,21074.88,8839.246,67.333333,6.30,7362.56
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,30485.0,...,8441.58,66.923333,6.5,7784.06,30424.0,21411.54,8194.537,66.000000,6.30,7784.06


In [157]:
# master_df["gas_per_mw"] = ( master_df["hourly_gas_burn"] / master_df["hourly_mw"]) # hourly_mw doesn't exist as a column. Assuming it should be hourly_site_gen_mw
master_df["gas_per_mw"] = ( master_df["hourly_gas_burn"] / master_df["hourly_site_gen_mw"])

master_df["gas_per_mw"] = master_df["gas_per_mw"].replace([np.inf, -np.inf], np.nan)

In [159]:
# Caleb Check
print(master_df.shape)
master_df.head()

(87520, 22)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,temperature,wind_speed,total_outages,load_actual,net_load_actual,wind_actual,temperature_actual,wind_speed_actual,total_outages_actual,gas_per_mw
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,33629.0,...,71.900000,6.5,7362.56,33707.0,24844.94,8511.573,69.666667,5.15,7362.56,44.337705
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,32426.0,...,70.640000,6.2,7362.56,32270.0,22600.18,8509.053,68.000000,5.70,7362.56,44.337705
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0,0.9475,31385.0,...,69.740000,6.2,7362.56,31254.0,21481.27,8834.246,67.666667,6.30,7362.56,44.337705
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,30679.0,...,68.840000,5.9,7362.56,30549.0,21074.88,8839.246,67.333333,6.30,7362.56,44.337705
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,30485.0,...,66.923333,6.5,7784.06,30424.0,21411.54,8194.537,66.000000,6.30,7784.06,44.337705


In [160]:
# Caleb Notes: Filling in missing actual values with forecasted values
master_df["load_final"] = master_df["load_actual"].fillna(master_df["load"])
master_df["wind_final"] = master_df["wind_actual"].fillna(master_df["wind"])
master_df["temperature_final"] = master_df["temperature_actual"].fillna(master_df["temperature"])


In [168]:
#clean and prep
master_df["hourly_gas_burn"] = pd.to_numeric(master_df["hourly_gas_burn"], errors="coerce")

# remove fake zeros (look into this)
# Caleb Note: How do we know these are 'fake'?
master_df.loc[ master_df["hourly_gas_burn"] == 0, "hourly_gas_burn"] = np.nan

master_df = master_df.sort_values(["site", "datetime"])


In [170]:
# Caleb Check
print(master_df.shape)
master_df.head()

(87520, 31)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,gas_per_mw,load_final,wind_final,temperature_final,hour,day_of_week,month,gas_lag_1,gas_lag_24,gas_roll_24
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,33629.0,...,44.337705,33707.0,8511.573,69.666667,1,2,7,NaN,NaN,NaN
1,2024-07-24 02:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,32426.0,...,44.337705,32270.0,8509.053,68.000000,2,2,7,1684.832787,NaN,NaN
2,2024-07-24 03:00:00,2024-07-23,CGS,305.0,37.9,13523.0,1680.399016,40.0,0.9475,31385.0,...,44.337705,31254.0,8834.246,67.666667,3,2,7,1689.266557,NaN,NaN
3,2024-07-24 04:00:00,2024-07-23,CGS,305.0,38.1,13523.0,1689.266557,40.0,0.9525,30679.0,...,44.337705,30549.0,8839.246,67.333333,4,2,7,1680.399016,NaN,NaN
4,2024-07-24 05:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.9500,30485.0,...,44.337705,30424.0,8194.537,66.000000,5,2,7,1689.266557,NaN,NaN


In [171]:
#adding time features
master_df["hour"] = master_df["datetime"].dt.hour
master_df["day_of_week"] = master_df["datetime"].dt.dayofweek
master_df["month"] = master_df["datetime"].dt.month

In [173]:
#adding lag features
master_df["gas_lag_1"] = master_df.groupby("site")["hourly_gas_burn"].shift(1)

master_df["gas_lag_24"] = master_df.groupby("site")["hourly_gas_burn"].shift(24)

master_df["gas_roll_24"] = master_df.groupby("site")["hourly_gas_burn"].transform(lambda x: x.shift(1).rolling(24).mean())
master_df.to_csv(r'./output-data/time and lag features check.csv', index=False)


In [176]:
#Caleb Check
# Caleb Check
print(master_df.shape)
master_df.head(1)

(87520, 31)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,gas_per_mw,load_final,wind_final,temperature_final,hour,day_of_week,month,gas_lag_1,gas_lag_24,gas_roll_24
0,2024-07-24 01:00:00,2024-07-23,CGS,305.0,38.0,13523.0,1684.832787,40.0,0.95,33629.0,...,44.337705,33707.0,8511.573,69.666667,1,2,7,NaN,NaN,NaN


In [ ]:
## data set for model training
# Caleb note: What's the purpose of this section? 

model_df = master_df.copy()

model_df = model_df.dropna(subset=["hourly_gas_burn", "gas_lag_1", "gas_lag_24", "gas_roll_24"])

#model_df.to_csv(r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Training Datasets\Model Input Data YES Forecast Added and shift update.csv", index=False)

print(model_df.shape)


(40588, 31)


In [ ]:
# Caleb Check
# We're losing a large number of records with the calling of .dropna(). Going from 87K to 40K records
print(model_df.shape)
model_df.head()

(40588, 31)


,datetime,gas_day,site,daily_site_gen_mw,hourly_site_gen_mw,daily_gas_burn,hourly_gas_burn,availability_mw,utilization,load,...,gas_per_mw,load_final,wind_final,temperature_final,hour,day_of_week,month,gas_lag_1,gas_lag_24,gas_roll_24
24,2024-07-25 01:00:00,2024-07-24,CGS,1215.8,39.5,13343.0,433.499342,40.0,0.9875,35327.0,...,10.974667,34825.0,13230.473,74.000000,1,3,7,463.130943,1684.832787,972.996319
25,2024-07-25 02:00:00,2024-07-24,CGS,1215.8,40.2,13343.0,441.181609,40.0,1.0050,33788.0,...,10.974667,33314.0,12927.840,73.333333,2,3,7,433.499342,1689.266557,920.857426
26,2024-07-25 03:00:00,2024-07-24,CGS,1215.8,40.1,13343.0,440.084142,40.0,1.0025,32795.0,...,10.974667,32207.0,12004.461,72.000000,3,3,7,441.181609,1680.399016,868.853886
27,2024-07-25 04:00:00,2024-07-24,CGS,1215.8,40.2,13343.0,441.181609,40.0,1.0050,32064.0,...,10.974667,31509.0,11042.648,72.666667,4,3,7,440.084142,1689.266557,817.174100
28,2024-07-25 05:00:00,2024-07-24,CGS,1215.8,40.1,13343.0,440.084142,40.0,1.0025,31745.0,...,10.974667,31239.0,9949.230,71.000000,5,3,7,441.181609,1684.832787,765.170560


In [ ]:
#Caleb Note: Again what is the purpose of this block

#final dataset
model_df = master_df.copy()


print(f'Dimensions of model_df before calling dropna(): {model_df.shape}')
model_df = model_df.dropna(subset=[ "hourly_gas_burn", "gas_lag_1", "gas_lag_24", "gas_roll_24"])
model_df = model_df.sort_values(["site", "datetime"])

print(f'Dimensions of model_df after calling dropna(): {model_df.shape}')


Dimensions of model_df before calling dropna(): (87520, 31)
Dimensions of model_df after calling dropna(): (40588, 31)


In [191]:
# Caleb Check
# how many nulls are from each column?
print(master_df[[ "hourly_gas_burn", "gas_lag_1", "gas_lag_24", "gas_roll_24"]].isna().sum())

# ~30K for for gas urn and lags. !47K for the roll. These fields need to be reevaluated. Should they even be removed in the first place


hourly_gas_burn    29906
gas_lag_1          29908
gas_lag_24         29966
gas_roll_24        46605
dtype: int64


In [180]:
model_export = model_df.copy()

model_export = model_export.sort_values(  ["site", "datetime"])

#model_export.to_csv( r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\MASTER_MERGED_DATA.csv", index=False)

print("Exported MODEL_INPUT_DATA.csv")
print("Rows:", len(model_export))
print("Columns:", len(model_export.columns))


print("\n===== FINAL MODEL SUMMARY =====")

print(model_df.groupby("site").size())

print("\nMissing Values")
print(model_df[[ "hourly_gas_burn", "availability_mw", "utilization", "gas_lag_1", "gas_lag_24", "gas_roll_24" ]].isna().sum())

print("\nGas Per MW")
print(model_df.groupby("site")["gas_per_mw"].describe())

model_df["availability_mw"] = model_df.groupby("site")["availability_mw"].transform( lambda x: x.fillna(x.median()))



#listing the features
features = ["load_final", "wind_final", "temperature_final", "availability_mw", "utilization", "hour", "day_of_week", "month", "gas_lag_1", "gas_lag_24", "gas_roll_24"]

Exported MODEL_INPUT_DATA.csv
Rows: 40588
Columns: 31

===== FINAL MODEL SUMMARY =====
site
CGS     6146
DCS     6708
GGS      496
LCS    11793
PGS    15445
dtype: int64

Missing Values
hourly_gas_burn       0
availability_mw    2647
utilization        3155
gas_lag_1             0
gas_lag_24            0
gas_roll_24           0
dtype: int64

Gas Per MW
        count       mean       std       min       25%        50%        75%  \
site                                                                           
CGS    6146.0   9.924028  0.511396  4.355828  9.605953  10.024845  10.195838   
DCS    6708.0   7.457466  0.525662  0.750000  7.374625   7.489939   7.585133   
GGS     496.0   9.077466  0.904722  3.411483  8.898609   9.128543   9.643966   
LCS   11793.0   9.922617  0.643223  7.239783  9.701680   9.899322  10.093271   
PGS   15445.0  14.214115  7.071197  3.108854  9.431149  10.169172  19.392402   

            max  
site             
CGS   12.953002  
DCS   13.308052  
GGS   11.673

In [212]:
#train models per site
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

models = {}
results = {}

split_date = "2025-01-01"

for site in model_df["site"].unique():

    print(f"\nTraining model for: {site}")

    site_df = model_df[model_df["site"] == site].sort_values("datetime")

    split_idx = int(len(site_df) * 0.8)

    train = site_df.iloc[:split_idx]
    test  = site_df.iloc[split_idx:]


    print(site, "train:", len(train), "test:", len(test))
    
    
    
    X_train = train[features]
    y_train = train["hourly_gas_burn"]

    X_test = test[features]
    y_test = test["hourly_gas_burn"]

    model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)

    print(f"{site} MAE:", mae)

    models[site] = model
    results[site] = mae



Training model for: CGS
CGS train: 4916 test: 1230
CGS MAE: 22.429896283363117

Training model for: DCS
DCS train: 5366 test: 1342
DCS MAE: 137.69809408590393

Training model for: GGS
GGS train: 396 test: 100
GGS MAE: 131.77411751255238

Training model for: LCS
LCS train: 9434 test: 2359
LCS MAE: 64.95312665858197

Training model for: PGS
PGS train: 12356 test: 3089
PGS MAE: 527.2530805579727


In [213]:
#Caleb Check on master_df
# note some of this may be getting deflated due to 0s not being taken out
master_df.loc[:, ['site', 'hourly_gas_burn']].groupby('site').describe()

hourly_gas_burn                                                     \
               count         mean          std         min          25%   
site                                                                      
CGS           6964.0   653.834578   137.232023    5.296085   605.239860   
DCS          11450.0  1628.833013   553.015218    7.654412  1294.765296   
GGS           6171.0   644.399125   317.579923    0.864223   565.314407   
LCS          15960.0  1095.989035   565.213816    0.922212   674.957451   
PGS          17069.0  2273.176167  1826.584514 -573.496842  1040.194622   

                                              
              50%          75%           max  
site                                          
CGS    634.009859   728.903563   2175.971891  
DCS   1707.659336  2044.866249   7376.825397  
GGS    667.661511   801.357355  11036.000000  
LCS   1088.233880  1501.673134   5587.098647  
PGS   1645.970932  3146.870781  10780.756072

In [214]:
# back test validation of the forecast model 

for site in models:
    site_df = model_df[model_df["site"] == site]
    preds = models[site].predict(site_df[features])
    
    mae = mean_absolute_error(site_df["hourly_gas_burn"], preds)
    print(f"{site} Backtest MAE:", mae)
    
    
    
# feature /drivers
for site in models:
    importances = pd.Series(models[site].feature_importances_, index=features).sort_values(ascending=False)

    print(f"\nTop drivers for {site}")
    print(importances.head(10))



print("Rows in master_df:", len(master_df))
print("Rows in model_df:", len(model_df))
print("Unique sites:", model_df["site"].unique())


CGS Backtest MAE: 10.522715558306707
DCS Backtest MAE: 48.669185573572406
GGS Backtest MAE: 27.769073276047557
LCS Backtest MAE: 34.438528936548494
PGS Backtest MAE: 188.76450700905954

Top drivers for CGS
gas_lag_1            0.576019
hour                 0.127106
availability_mw      0.113888
utilization          0.087606
month                0.027299
temperature_final    0.021794
gas_lag_24           0.014032
gas_roll_24          0.011656
wind_final           0.008071
day_of_week          0.006844
dtype: float32

Top drivers for DCS
utilization          0.778504
gas_lag_1            0.099725
availability_mw      0.070379
hour                 0.009258
gas_roll_24          0.008587
month                0.006811
wind_final           0.005991
temperature_final    0.005885
day_of_week          0.005702
load_final           0.005599
dtype: float32

Top drivers for GGS
utilization          0.650230
availability_mw      0.130449
temperature_final    0.126829
gas_lag_1            0.022587
mo

In [ ]:
#create future timestamp

horizon = 168 # One week at hourly grain

now = pd.Timestamp.now()

# getting the gas day start time for the current calendar day
# now() gets current day => get midnight for the in scope gas day => shift forward to start of gas day
forecast_start = ((now - pd.Timedelta(hours=9)).normalize() + pd.Timedelta(hours=9))

future_dates = pd.date_range(start=forecast_start, periods=horizon, freq="h")

print(now)
print(forecast_start)
print(future_dates)



2026-07-27 15:37:45.312525
2026-07-27 09:00:00
DatetimeIndex(['2026-07-27 09:00:00', '2026-07-27 10:00:00',
               '2026-07-27 11:00:00', '2026-07-27 12:00:00',
               '2026-07-27 13:00:00', '2026-07-27 14:00:00',
               '2026-07-27 15:00:00', '2026-07-27 16:00:00',
               '2026-07-27 17:00:00', '2026-07-27 18:00:00',
               ...
               '2026-08-02 23:00:00', '2026-08-03 00:00:00',
               '2026-08-03 01:00:00', '2026-08-03 02:00:00',
               '2026-08-03 03:00:00', '2026-08-03 04:00:00',
               '2026-08-03 05:00:00', '2026-08-03 06:00:00',
               '2026-08-03 07:00:00', '2026-08-03 08:00:00'],
              dtype='datetime64[us]', length=168, freq='h')


In [223]:
#ADD in forward looking data
#build future_df 

sites = model_df["site"].unique()

future_df = pd.MultiIndex.from_product([future_dates, sites], names=["datetime","site"]).to_frame(index=False)
print(f'future df dimensions: {future_df.shape}')
future_df.head()

future df dimensions: (840, 2)


,datetime,site
0,2026-07-27 09:00:00,CGS
1,2026-07-27 09:00:00,DCS
2,2026-07-27 09:00:00,GGS
3,2026-07-27 09:00:00,LCS
4,2026-07-27 09:00:00,PGS


In [224]:
future_df["hour"] = future_df["datetime"].dt.hour
future_df["day_of_week"] = future_df["datetime"].dt.dayofweek
future_df["month"] = future_df["datetime"].dt.month

future_df.head()


,datetime,site,hour,day_of_week,month
0,2026-07-27 09:00:00,CGS,9,0,7
1,2026-07-27 09:00:00,DCS,9,0,7
2,2026-07-27 09:00:00,GGS,9,0,7
3,2026-07-27 09:00:00,LCS,9,0,7
4,2026-07-27 09:00:00,PGS,9,0,7


In [225]:
# load and process availability
import os
from glob import glob

forward_folder = r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\Transposed Forward Looking Data"

print("Folder exists:", os.path.exists(forward_folder))

try:
    print("Directory contents:")
    for f in os.listdir(forward_folder):
        print(f)
except Exception as e:
    print("Error accessing folder:", e)

# getting the forward looking HEL
files = [ f for f in glob(os.path.join(forward_folder, "*forward*transposed*.xls*"))
    if not os.path.basename(f).startswith("~$")]


if not files:
    raise FileNotFoundError("No forward-looking files matched pattern")

# grab the most recent file
forward_file = max(files, key=os.path.getctime)


xls = pd.ExcelFile(forward_file)

sheet_name = [s for s in xls.sheet_names if "transposed" in s.lower()][0]

forward_df = pd.read_excel(xls, sheet_name=sheet_name)



Folder exists: True
Directory contents:
excel forward looking transposed (6-26 thru 7-03).xlsx
excel forward looking transposed (6-29 thru 7-06).xlsx
excel forward looking transposed (6-30 thru 7-07).xlsx
excel forward looking transposed (7-02 thru 7-09).xlsx
excel forward looking transposed (7-07 thru 7-14).xlsx
excel forward looking transposed (7-09 thru 7-16).xlsx
excel forward looking transposed (7-10 thru 7-17).xlsx
excel forward looking transposed (7-13 thru 7-20).xlsx
excel forward looking transposed (7-17 thru 7-24).xlsx
~$excel forward looking transposed (7-17 thru 7-24).xlsx


In [226]:
forward_df.head()

,datetime,CGS1 - High Effective Limit,DCS1 - High Effective Limit,GGS1 - High Effective Limit,GGS2 - High Effective Limit,LCS1 - High Effective Limit,LCS2 - High Effective Limit,LCS3 - High Effective Limit,LCS4 - High Effective Limit,LCS5 - High Effective Limit,...,PGS21 - High Effective Limit,PGS22 - High Effective Limit,PGS31 - High Effective Limit,PGS32 - High Effective Limit,PGS33 - High Effective Limit,PGS34 - High Effective Limit,PGS35 - High Effective Limit,PGS36 - High Effective Limit,PGS4 - High Effective Limit,PGS5 - High Effective Limit
0,2026-07-17 00:00:00,0,297,58,58,32,39,39,39,39,...,8.9,8.9,18.6,18.6,0.0,18.6,18.6,18.6,208.00,208.00
1,2026-07-17 01:00:00,0,297,58,58,32,39,39,39,39,...,8.9,8.9,18.6,18.6,0.0,18.6,18.6,18.6,209.92,209.92
2,2026-07-17 02:00:00,0,297,58,58,32,39,39,39,39,...,8.9,8.9,18.6,18.6,0.0,18.6,18.6,18.6,210.66,210.66
3,2026-07-17 03:00:00,0,297,58,58,32,39,39,39,39,...,8.9,8.9,18.6,18.6,0.0,18.6,18.6,18.6,211.38,211.38
4,2026-07-17 04:00:00,0,297,58,58,32,39,39,39,39,...,8.9,8.9,18.6,18.6,0.0,18.6,18.6,18.6,212.10,212.10


In [ ]:

# cleaning the file
forward_df.columns = [col.replace(" - High Effective Limit", "").strip()
    for col in forward_df.columns]

forward_df["datetime"] = pd.to_datetime(forward_df["datetime"])

# melting data into a format that can be used
forward_long = forward_df.melt( id_vars=["datetime"], var_name="unit", value_name="availability_mw")



def map_unit_to_site(u):
    if str(u).startswith("DCS"):
        return "DCS"
    elif str(u).startswith("LCS"):
        return "LCS"
    elif str(u).startswith("PGS"):
        return "PGS"
    elif str(u).startswith("GGS"):
        return "GGS"
    elif str(u).startswith("CGS"):
        return "CGS"
    return None

forward_long["site"] = forward_long["unit"].apply(map_unit_to_site)
forward_long = forward_long.dropna(subset=["site"])

forward_site = (forward_long.groupby(["datetime","site"], as_index=False)["availability_mw"].sum())


# merge availability 
future_df = future_df.merge(forward_site, on=["datetime","site"], how="left")

future_df["availability_mw"] = (future_df["availability_mw"].ffill().bfill())


# add utilization
util_lookup = master_df.groupby(["site","hour"])["utilization"].mean()

future_df = future_df.merge(util_lookup.rename("utilization"), on=["site","hour"], how="left")


# merge forecast data
future_df = future_df.merge(df_forecast_clean, on="datetime", how="left")

# cleaning up the naming 
def safe_final(df, actual, forecast):
    if actual in df.columns:
        return df[actual].fillna(df[forecast])
    return df[forecast]

future_df["load_final"] = safe_final(future_df, "load_actual", "load")
future_df["wind_final"] = safe_final(future_df, "wind_actual", "wind")
future_df["temperature_final"] = safe_final(future_df, "temperature_actual", "temperature")









#building forecasting loop
#add site capacity CHECK THESE NUMBERS
site_capacity = {"CGS": 200, "DCS": 300, "LCS": 800, "PGS": 500, "GGS": 150}

gas_per_mw_lookup = master_df.groupby(["site", "hour"])["gas_per_mw"].median()

# forecasting loop
all_forecasts = []

for site in models.keys():

    print(f"\nForecasting for {site}")

    model = models[site]

    site_hist = master_df[master_df["site"] == site].sort_values("datetime")
    site_hist = site_hist[site_hist["datetime"] < future_dates.min()]

    history = site_hist.tail(48).copy()
    future_preds = []

    for dt in future_dates:

        new_row = { "datetime": dt, "site": site, "hour": dt.hour, "day_of_week": dt.dayofweek, "month": dt.month}

        api_row = future_df[
            (future_df["datetime"] == dt) & 
            (future_df["site"] == site)]

        # forecast inputs
        if len(api_row) > 0:
            new_row["load_final"] = api_row["load_final"].values[0]
            new_row["wind_final"] = api_row["wind_final"].values[0]
            new_row["temperature_final"] = api_row["temperature_final"].values[0]
            new_row["availability_mw"] = api_row["availability_mw"].values[0]
        else:
            new_row["load_final"] = history["load_final"].iloc[-1]
            new_row["wind_final"] = history["wind_final"].iloc[-1]
            new_row["temperature_final"] = history["temperature_final"].iloc[-1]
            new_row["availability_mw"] = history["availability_mw"].iloc[-1]

        # utilization
        lookup_val = util_lookup.get((site, dt.hour), np.nan)

        if not pd.isna(lookup_val):
            new_row["utilization"] = lookup_val
        else:
            new_row["utilization"] = history["utilization"].dropna().iloc[-1]





#look into this as well 
        # lags
        new_row["gas_lag_1"] = history["hourly_gas_burn"].iloc[-1]
        new_row["gas_lag_24"] = history["hourly_gas_burn"].iloc[-24]
        new_row["gas_roll_24"] = history["hourly_gas_burn"].iloc[-24:].mean()

        # building the input
        X_pred = pd.DataFrame([new_row])[features]

        # prediction
        pred = model.predict(X_pred)[0]

        # availability constraint
        cap = site_capacity.get(site, 200)

        if new_row["availability_mw"] <= 1:
            pred = 0
        else:
            scale = min(new_row["availability_mw"] / cap, 1)
            pred *= scale





#something to look into 
        # minimum mw constraint
        gas_per_mw = gas_per_mw_lookup.get((site, dt.hour), 8)
        implied_mw = pred / gas_per_mw

        if implied_mw < 0:
            pred = 0

        new_row["predicted_gas_burn"] = pred

        # recursion
        history = pd.concat(
            [history, pd.DataFrame([{**new_row, "hourly_gas_burn": pred}])], ignore_index=True)

        future_preds.append(new_row)

    all_forecasts.append(pd.DataFrame(future_preds))




    
    
    

    #combine all sites
forecast_df = pd.concat(all_forecasts, ignore_index=True)



# adding gas day
forecast_df["gas_day"] = (pd.to_datetime(forecast_df["datetime"]) - pd.Timedelta(hours=9)).dt.date

# and gas_per_mw lookup 
gas_per_mw_lookup = master_df.groupby(["site", "hour"])["gas_per_mw"].median()

# add hour for merge 
forecast_df["hour"] = pd.to_datetime(forecast_df["datetime"]).dt.hour

# then merge gas_per_mw 
forecast_df = forecast_df.merge(gas_per_mw_lookup.rename("gas_per_mw"), on=["site", "hour"], how="left")

forecast_df["implied_mw"] = forecast_df["predicted_gas_burn"] / forecast_df["gas_per_mw"]



# final formatting here
forecast_df = forecast_df[["datetime", "gas_day", "site", "predicted_gas_burn", "gas_per_mw", "implied_mw"]]

# now daily aggregation
daily_forecast = (forecast_df.groupby(["gas_day", "site"], as_index=False)["predicted_gas_burn"].sum())

# and sort + format 
forecast_df = forecast_df.sort_values(["site", "datetime"])







#final formatting:
forecast_df = forecast_df.drop_duplicates(subset=["datetime", "site"])
forecast_df = forecast_df.sort_values(["site", "datetime"])
forecast_df["datetime"] = forecast_df["datetime"].dt.strftime("%Y-%m-%d %H:%M")


#export to excel
file_path = r"G:\Trading\Forecasts\Daily Gas Burn Forecast by Site\forecast with historicals checked.xlsx"

with pd.ExcelWriter(file_path, engine="openpyxl") as writer:

    for site in forecast_df["site"].unique():
        site_df = forecast_df[forecast_df["site"] == site]
        site_df.to_excel(writer, sheet_name=site, index=False)

print("File saved at:", file_path)